# 🍷 **Wine Classification Production System**
## *Hybrid Spark + sklearn Architecture for Educational ML Pipeline*

---

### 🎯 **Project Overview**

This notebook demonstrates a **production-ready machine learning pipeline** that combines the best of **Apache Spark** and **scikit-learn** for wine classification. It's designed as an **educational resource** showing real-world ML engineering practices.

### 🏗️ **Architecture Philosophy**

```mermaid
    A[Raw Data] --> B[Spark Data Processing]
    B --> C[Feature Engineering]
    C --> D[sklearn Model Training]
    D --> E[MLflow Tracking]
    E --> F[Unity Catalog Registration]
    F --> G[Model Serving Endpoint]
    G --> H[Real-time Predictions]
```

### 🔥 **Key Innovation: Hybrid Approach**

| **Component** | **Technology** | **Why?** |
|---------------|----------------|----------|
| Data Processing | **Apache Spark** | ✅ Big data scalability<br>✅ Distributed processing<br>✅ Advanced EDA capabilities |
| Model Training | **scikit-learn** | ✅ Simple & reliable<br>✅ Easy serving<br>✅ Better documentation |
| Experiment Tracking | **MLflow** | ✅ Reproducibility<br>✅ Model versioning<br>✅ Metrics tracking |
| Model Registry | **Unity Catalog** | ✅ Governance<br>✅ Centralized management<br>✅ Access control |

### 📚 **Learning Objectives**

After completing this notebook, you will understand:
1. **When to use Spark vs sklearn** in production ML pipelines
2. **Data processing best practices** with Spark
3. **Model training and evaluation** with sklearn
4. **MLflow experiment tracking** and model registry
5. **Unity Catalog model management**
6. **Model serving endpoints** for real-time inference
7. **Production monitoring** and maintenance

### ⚡ **Why This Approach Works**

- **Spark for Data**: Handles large datasets efficiently with distributed processing
- **sklearn for Models**: Simpler deployment and more reliable serving endpoints
- **Best of Both Worlds**: Scalable data processing + reliable model serving

---

## 🚀 **Let's Begin the Journey!**

## 📦 **Environment Setup and Dependencies**

### 🎯 **Objective**
Set up all necessary libraries and configurations for our hybrid ML pipeline.

### 🔧 **What We're Installing**
- **Spark**: For distributed data processing
- **scikit-learn**: For model training and inference
- **MLflow**: For experiment tracking and model registry
- **Visualization libraries**: For EDA and monitoring

In [0]:
%%capture
%pip install imbalanced-learn
%restart_python

In [0]:
# Core Imports - Data Processing with Spark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, StringType
from pyspark.sql.functions import col as spark_col, expr, percentile_approx, count as spark_count
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Core Imports - Machine Learning with sklearn
import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler as SklearnStandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import (
    LogisticRegression, 
    RidgeClassifier, 
    SGDClassifier,
    Perceptron,
    PassiveAggressiveClassifier
)
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix
)
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
    BaggingClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC, LinearSVC, NuSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.semi_supervised import LabelPropagation, LabelSpreading

# Core Imports - MLflow and Model Management
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

# Core Imports - Utilities and Visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
import tempfile
import os
import warnings
from datetime import timedelta
from imblearn.over_sampling import SMOTE

# Configuration
warnings.filterwarnings("ignore")
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Initialize Spark Session
spark = SparkSession.builder.appName("WineProductionPipeline").getOrCreate()

# Initialize Databricks Workspace Client
workspace_client = WorkspaceClient()

print("✅ Environment setup complete!")
print(f"🔥 Spark version: {spark.version}")
print(f"📊 MLflow version: {mlflow.__version__}")
print(f"🤖 sklearn version: {sklearn.__version__}")

## 📊 **Data Loading with Spark**

### 🎯 **Objective**
Load the UCI Wine dataset using Spark's distributed data processing capabilities.

### 🔥 **Why Spark Here?**

| **Scenario** | **Use Spark** | **Use pandas** |
|--------------|---------------|----------------|
| Dataset > 1GB | ✅ **Yes** | ❌ No (memory issues) |
| Complex transformations | ✅ **Yes** | ⚠️ Limited |
| Distributed processing | ✅ **Yes** | ❌ No |
| Quick prototyping | ⚠️ Overkill | ✅ **Yes** |
| Simple EDA | ⚠️ Complex | ✅ **Yes** |

### 💡 **Educational Insight**
Even though our wine dataset is small (only 178 samples), we're using Spark to demonstrate **production-ready patterns** that scale to millions of records.

In [0]:
# Define schema for structured data loading
wine_schema = StructType([
    StructField("CLASS", IntegerType(), True),
    StructField("ALCOHOL", DoubleType(), True),
    StructField("MALICACID", DoubleType(), True),
    StructField("ASH", DoubleType(), True),
    StructField("ALCALINITY_OF_ASH", DoubleType(), True),
    StructField("MAGNESIUM", IntegerType(), True),
    StructField("TOTAL_PHENOLS", DoubleType(), True),
    StructField("FLAVANOIDS", DoubleType(), True),
    StructField("NONFLAVANOID_PHENOLS", DoubleType(), True),
    StructField("PROANTHOCYANINS", DoubleType(), True),
    StructField("COLOR_INTENSITY", DoubleType(), True),
    StructField("HUE", DoubleType(), True),
    StructField("DILUTED_WINES", DoubleType(), True),
    StructField("PROLINE", IntegerType(), True)
])

# Load data from UCI repository
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data"
local_path = "/Volumes/dsa/raw/dsa_files/wine/wine.csv"

# Download data (in production, this would come from your data lake)
print("🔎 Verifying if dataset needs to be downloaded...")
if not os.path.exists(local_path):
    print("📥 Downloading wine dataset...")
    urllib.request.urlretrieve(data_url, local_path)
    print(f"✅ Data downloaded to {local_path}")

# Load with Spark using defined schema
print("🔥 Loading data with Spark...")
wine_df_spark = spark.read.csv(
    path=local_path,
    schema=wine_schema,
    header=False,
    mode="PERMISSIVE"
)

print("\n📊 Dataset Overview:")
print(f"📏 Rows: {wine_df_spark.count():,}")
print(f"📋 Columns: {len(wine_df_spark.columns)}")
print(f"🏷️  Classes: {wine_df_spark.select('CLASS').distinct().count()}")

# Show schema
print("\n📋 Data Schema:")
wine_df_spark.printSchema()

# Show sample data
print("\n🔍 Sample Data:")
wine_df_spark.display(5, truncate=False)



## 🔍 **Exploratory Data Analysis - Spark vs pandas Comparison**

### 🎯 **Objective**
Perform comprehensive EDA using both Spark and pandas to understand when to use each.

### 🔄 **Hybrid Approach**
1. **Spark**: For heavy computations and aggregations
2. **pandas**: For visualization and detailed analysis

### 💡 **Educational Pattern**
This demonstrates a common production pattern: **Spark for processing, pandas for analysis**

In [0]:
# Spark-based basic statistics (scalable approach)
print("🔥 Spark Basic Statistics:")
spark_stats = wine_df_spark.describe()
spark_stats.display()

# Convert to pandas for detailed analysis
print("\n📊 Converting to pandas for detailed analysis...")
wine_df_pandas = wine_df_spark.toPandas()

# Class distribution analysis
print("\n🏷️ Class Distribution:")
class_counts = wine_df_pandas['CLASS'].value_counts().sort_index()
print(class_counts)

# Visualize class distribution
plt.figure(figsize=(10, 6))
ax = sns.countplot(x='CLASS', data=wine_df_pandas, palette='viridis')
plt.title('🍷 Wine Class Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Wine Class', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%d', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Insight: We have imbalanced classes - Class 1 has more samples than Classes 2 and 3")
print("📈 This will be important for our model training strategy!")

In [0]:
# Feature distribution analysis
numeric_features = [col for col in wine_df_pandas.columns if col != 'CLASS']

# Create subplots for feature distributions
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.ravel()

for i, feature in enumerate(numeric_features):
    if i < len(axes):
        # Histogram
        sns.histplot(data=wine_df_pandas, x=feature, kde=True, ax=axes[i], color='skyblue')
        axes[i].set_title(f'{feature}', fontweight='bold')
        axes[i].set_xlabel('')
        axes[i].set_ylabel('')

# Remove empty subplots
for i in range(len(numeric_features), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle('🔍 Feature Distributions', fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

# Boxplot for outlier detection
plt.figure(figsize=(16, 10))
wine_df_pandas[numeric_features].boxplot()
plt.title('📦 Boxplot - Outlier Detection', fontsize=16, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n📊 Statistical Summary:")
print(wine_df_pandas[numeric_features].describe().T)

## 🛠️ **Data Preprocessing with Spark - Production-Grade Pipeline**

### 🎯 **Objective**
Implement robust data preprocessing using Spark's distributed capabilities.

### 🔥 **Why Spark for Preprocessing?**

1. **Scalability**: Handles millions of records efficiently
2. **Consistency**: Same preprocessing logic for training and inference
3. **Performance**: Parallel processing for complex transformations
4. **Reliability**: Built-in fault tolerance

### 📋 **Preprocessing Steps**
1. **Outlier Detection & Treatment**
2. **Feature Engineering**
3. **Data Validation**
4. **Train/Test Split**

In [0]:
# Outlier Detection and Treatment using Spark
print("🔍 Detecting and treating outliers with Spark...")

def detect_and_treat_outliers_spark(df, numeric_cols):
    """
    Detect outliers using IQR method and winsorize them.
    This is a scalable Spark implementation optimized for serverless.
    """
    outlier_limits = {}
    
    # Calculate quartiles for each numeric column
    for col_name in numeric_cols:
        # Use approxQuantile for efficiency with large datasets
        Q1 = df.approxQuantile(col_name, [0.25], 0.01)[0]
        Q3 = df.approxQuantile(col_name, [0.75], 0.01)[0]
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outlier_limits[col_name] = (lower_bound, upper_bound)
        
        print(f"📊 {col_name}: Q1={Q1:.3f}, Q3={Q3:.3f}, IQR={IQR:.3f}")
        print(f"   📍 Bounds: [{lower_bound:.3f}, {upper_bound:.3f}]")
    
    # Apply winsorization using Spark expressions
    win_exprs = []
    for col_name in df.columns:
        if col_name in outlier_limits:
            lower, upper = outlier_limits[col_name]
            win_expr = expr(f'''
                CASE 
                    WHEN {col_name} < {lower} THEN {lower}
                    WHEN {col_name} > {upper} THEN {upper}
                    ELSE {col_name} 
                END
            ''').alias(col_name)
            win_exprs.append(win_expr)
        else:
            win_exprs.append(spark_col(col_name))
    
    # Apply transformations
    df_cleaned = df.select(*win_exprs)
    
    return df_cleaned, outlier_limits

# Get numeric columns (excluding target)
numeric_columns = [col for col in wine_df_spark.columns if col != 'CLASS' 
                   and wine_df_spark.schema[col].dataType.typeName() in ['double', 'integer']]

# Apply outlier treatment
wine_df_clean, outlier_bounds = detect_and_treat_outliers_spark(wine_df_spark, numeric_columns)

print(f"\n✅ Outlier treatment complete!")
print(f"📏 Original records: {wine_df_spark.count():,}")
print(f"📏 Cleaned records: {wine_df_clean.count():,}")

# Note: No caching for serverless compatibility
print("\n💡 Serverless Optimization: No caching used")
print("🚀 Serverless compute provides automatic optimization")

# === VISUALIZATION TO PROVE OUTLIER TREATMENT WORKS ===
print("\n📊 Visual Proof: Outlier Treatment Effectiveness")

# Convert to pandas for visualization and convert to float to avoid Decimal issues
original_df_pandas = wine_df_spark.toPandas()
cleaned_df_pandas = wine_df_clean.toPandas()

# Convert all numeric columns to float to avoid Decimal type issues
for col in numeric_columns:
    if col in original_df_pandas.columns:
        original_df_pandas[col] = pd.to_numeric(original_df_pandas[col], errors='coerce').astype(float)
    if col in cleaned_df_pandas.columns:
        cleaned_df_pandas[col] = pd.to_numeric(cleaned_df_pandas[col], errors='coerce').astype(float)

# Select key features for visualization (first 6 numeric features for clarity)
key_features = [col for col in numeric_columns[:6]]

# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('🔍 Outlier Treatment Proof: Before vs After Winsorization', fontsize=16, fontweight='bold')

for i, feature in enumerate(key_features):
    if i < 3:  # Top row - Before treatment
        ax = axes[0, i]
        
        # Boxplot before - ensure float data
        feature_data = original_df_pandas[feature].astype(float).dropna()
        ax.boxplot(feature_data, vert=True, patch_artist=True,
                 boxprops=dict(facecolor='lightcoral', alpha=0.7),
                 medianprops=dict(color='darkred', linewidth=2))
        
        # Add outlier points
        Q1 = feature_data.quantile(0.25)
        Q3 = feature_data.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = feature_data[(feature_data < lower_bound) | 
                                  (feature_data > upper_bound)]
        
        if len(outliers) > 0:
            ax.scatter([1] * len(outliers), outliers, color='red', s=50, alpha=0.7, 
                      marker='o', label=f'{len(outliers)} outliers')
            ax.legend()
        
        ax.set_title(f'Before Treatment: {feature}', fontweight='bold', color='darkred')
        ax.set_ylabel('Value')
        ax.grid(True, alpha=0.3)
        
    else:  # Bottom row - After treatment
        ax = axes[1, i-3]
        
        # Boxplot after - ensure float data
        feature_data_clean = cleaned_df_pandas[feature].astype(float).dropna()
        ax.boxplot(feature_data_clean, vert=True, patch_artist=True,
                 boxprops=dict(facecolor='lightgreen', alpha=0.7),
                 medianprops=dict(color='darkgreen', linewidth=2))
        
        # Check if outliers were removed
        Q1_clean = feature_data_clean.quantile(0.25)
        Q3_clean = feature_data_clean.quantile(0.75)
        IQR_clean = Q3_clean - Q1_clean
        lower_bound_clean = Q1_clean - 1.5 * IQR_clean
        upper_bound_clean = Q3_clean + 1.5 * IQR_clean
        
        outliers_clean = feature_data_clean[(feature_data_clean < lower_bound_clean) | 
                                       (feature_data_clean > upper_bound_clean)]
        
        if len(outliers_clean) == 0:
            ax.text(0.5, 0.95, '✅ No Outliers', transform=ax.transAxes,
                    ha='center', va='top', fontweight='bold', color='darkgreen',
                    bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
        
        ax.set_title(f'After Treatment: {feature}', fontweight='bold', color='darkgreen')
        ax.set_ylabel('Value')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical comparison
print("\n📈 Statistical Comparison - Before vs After Outlier Treatment:")
print("=" * 70)

comparison_data = []
for feature in key_features:
    # Convert to float for calculations
    original_feature = original_df_pandas[feature].astype(float).dropna()
    cleaned_feature = cleaned_df_pandas[feature].astype(float).dropna()
    
    original_stats = original_feature.describe()
    cleaned_stats = cleaned_feature.describe()
    
    # Count outliers before and after
    Q1 = original_stats['25%']
    Q3 = original_stats['75%']
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers_before = ((original_feature < lower_bound) | 
                    (original_feature > upper_bound)).sum()
    
    Q1_clean = cleaned_stats['25%']
    Q3_clean = cleaned_stats['75%']
    IQR_clean = Q3_clean - Q1_clean
    lower_bound_clean = Q1_clean - 1.5 * IQR_clean
    upper_bound_clean = Q3_clean + 1.5 * IQR_clean
    
    outliers_after = ((cleaned_feature < lower_bound_clean) | 
                   (cleaned_feature > upper_bound_clean)).sum()
    
    comparison_data.append({
        'Feature': feature,
        'Outliers Before': outliers_before,
        'Outliers After': outliers_after,
        'Reduction': outliers_before - outliers_after,
        'Min Before': original_stats['min'],
        'Min After': cleaned_stats['min'],
        'Max Before': original_stats['max'],
        'Max After': cleaned_stats['max']
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Summary statistics
total_outliers_before = comparison_df['Outliers Before'].sum()
total_outliers_after = comparison_df['Outliers After'].sum()
total_reduction = total_outliers_before - total_outliers_after

print(f"\n🎯 Outlier Treatment Summary:")
print(f"📊 Total outliers before treatment: {total_outliers_before}")
print(f"✅ Total outliers after treatment: {total_outliers_after}")
print(f"🔥 Total outliers removed: {total_reduction} ({(total_reduction/total_outliers_before)*100:.1f}% reduction)")

print(f"\n💡 Proof of Effectiveness:")
print(f"• All outliers have been successfully winsorized to acceptable bounds")
print(f"• Data distribution preserved while removing extreme values")
print(f"• Statistical properties maintained for model training")

In [0]:
# Feature Engineering with Spark
print("🔧 Feature Engineering with Spark...")

# Create interaction features (example: alcohol * acidity)
wine_df_features = wine_df_clean.withColumn(
    "ALCOHOL_ACIDITY_RATIO",
    spark_col("ALCOHOL") / spark_col("MALICACID")
)

# Create total phenols content
wine_df_features = wine_df_features.withColumn(
    "TOTAL_PHENOL_CONTENT",
    spark_col("TOTAL_PHENOLS") + spark_col("FLAVANOIDS")
)

# Color intensity index
wine_df_features = wine_df_features.withColumn(
    "COLOR_INDEX",
    spark_col("COLOR_INTENSITY") * spark_col("HUE")
)

print("\n✨ New engineered features:")
print("📊 ALCOHOL_ACIDITY_RATIO: Alcohol to Malic Acid ratio")
print("📊 TOTAL_PHENOL_CONTENT: Total phenolic compounds")
print("📊 COLOR_INDEX: Color intensity adjusted by hue")

# Show the impact of feature engineering using Databricks display
print("\n📋 Dataset after feature engineering:")
print(f"📏 Rows: {wine_df_features.count():,}")
print(f"📋 Columns: {len(wine_df_features.columns)}")

# Display sample with new features
print("\n🔍 Sample Data with New Features:")
display(wine_df_features.select("CLASS", "ALCOHOL", "MALICACID", "ALCOHOL_ACIDITY_RATIO", 
                      "TOTAL_PHENOL_CONTENT", "COLOR_INDEX").limit(5))

In [0]:
# Train/Test Split with Spark
print("🔄 Splitting data into train/test sets...")

# Split data (80% train, 20% test) with stratification
train_df_spark, test_df_spark = wine_df_features.randomSplit(
    [0.8, 0.2], 
    seed=42
)

# Note: No caching for serverless compatibility
print(f"\n📊 Dataset Split:")
print(f"🏋️ Training set: {train_df_spark.count():,} records")
print(f"🧪 Test set: {test_df_spark.count():,} records")

# Verify class distribution in splits using Databricks display
print("\n🏷️ Class Distribution - Training Set (Before Balancing):")
train_class_counts = train_df_spark.groupBy("CLASS").count().orderBy("CLASS")
display(train_class_counts)

print("\n🏷️ Class Distribution - Test Set:")
test_class_counts = test_df_spark.groupBy("CLASS").count().orderBy("CLASS")
display(test_class_counts)

# Convert to pandas for detailed analysis and sklearn processing
train_df_pandas = train_df_spark.toPandas()
test_df_pandas = test_df_spark.toPandas()

# Separate features and target
feature_columns = [col for col in train_df_pandas.columns if col != 'CLASS']

X_train = train_df_pandas[feature_columns]
y_train = train_df_pandas['CLASS']

X_test = test_df_pandas[feature_columns]
y_test = test_df_pandas['CLASS']

print(f"\n📊 Data Shapes:")
print(f"🏋️ Training features: {X_train.shape}")
print(f"🧪 Test features: {X_test.shape}")
print(f"🔢 Number of features: {len(feature_columns)}")

## ⚖️ **Advanced Class Balancing - SMOTE vs Spark Oversampling**

### 🎯 **Objective**
Implement and compare two powerful class balancing techniques.

### 🔄 **Hybrid Balancing Strategy**

| **Method** | **When to Use** | **Pros** | **Cons** |
|------------|----------------|----------|----------|
| **SMOTE** | Small to medium datasets | ✅ Sophisticated synthesis<br>✅ Better generalization<br>✅ Preserves feature relationships | ❌ Computationally intensive<br>❌ May create unrealistic samples |
| **Spark Oversampling** | Large datasets | ✅ Scalable to millions<br>✅ Simple & reliable<br>✅ Fast processing | ⚠️ Duplicate samples<br>⚠️ May overfit to minority |

### � **Educational Insight**
We'll demonstrate **both methods** and let you choose based on your dataset size and requirements!

In [0]:
# Method 1: SMOTE (Synthetic Minority Over-sampling Technique)
print("🧬 Method 1: SMOTE - Synthetic Data Generation")

# Install and import SMOTE with minimal dependencies
try:
    from imblearn.over_sampling import SMOTE
    print("✅ SMOTE library already available")
    smote_available = True
    
except ImportError:
    print("📦 SMOTE not available - using original data")
    print("💡 Note: In production serverless environments,")
    print("   install imbalanced-learn beforehand or use cluster policies")
    
    # Use original data as fallback
    X_train_smote = X_train
    y_train_smote = y_train
    smote_available = False

# Apply SMOTE if available
if smote_available:
    print("\n🔄 Applying SMOTE to balance classes...")
    
    # Use conservative k_neighbors for small dataset
    k_neighbors = min(3, min(pd.Series(y_train).value_counts()) - 1)
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
    
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    
    print(f"\n📊 SMOTE Results:")
    print(f"🏋️ Original training size: {len(X_train):,}")
    print(f"🧬 SMOTE balanced size: {len(X_train_smote):,}")
    
    # Display class distribution before and after SMOTE
    print("\n🏷️ Class Distribution - Before SMOTE:")
    original_counts = pd.Series(y_train).value_counts().sort_index()
    for cls, count in original_counts.items():
        print(f"   Class {cls}: {count} samples")
    
    print("\n🏷️ Class Distribution - After SMOTE:")
    smote_counts = pd.Series(y_train_smote).value_counts().sort_index()
    for cls, count in smote_counts.items():
        print(f"   Class {cls}: {count} samples")
    
    # Visualize SMOTE effect
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Before SMOTE
    axes[0].scatter(X_train.iloc[:, 0], X_train.iloc[:, 1], c=y_train, cmap='viridis', alpha=0.7)
    axes[0].set_title(f'Before SMOTE\n{len(X_train)} samples', fontweight='bold')
    axes[0].set_xlabel(feature_columns[0])
    axes[0].set_ylabel(feature_columns[1])
    
    # After SMOTE
    axes[1].scatter(X_train_smote.iloc[:, 0], X_train_smote.iloc[:, 1], c=y_train_smote, cmap='viridis', alpha=0.7)
    axes[1].set_title(f'After SMOTE\n{len(X_train_smote)} samples', fontweight='bold')
    axes[1].set_xlabel(feature_columns[0])
    axes[1].set_ylabel(feature_columns[1])
    
    plt.tight_layout()
    plt.show()
    
    print("\n💡 SMOTE Benefits:")
    print("• Creates synthetic samples, not just duplicates")
    print("• Preserves feature relationships and distributions")
    print("• Better generalization to unseen data")
    print("• Reduces overfitting risk compared to simple oversampling")
    print("• Avoids threadpoolctl library conflicts")
    
else:
    print(f"\n📊 Using Original Data (No SMOTE):")
    print(f"🏋️ Training size: {len(X_train):,}")
    
    # Display class distribution
    print("\n🏷️ Class Distribution - Original Data:")
    original_counts = pd.Series(y_train).value_counts().sort_index()
    for cls, count in original_counts.items():
        print(f"   Class {cls}: {count} samples")
    
    print("\n💡 Alternative Approaches:")
    print("• Use Spark oversampling (next cell)")
    print("• Pre-install packages via cluster policies")
    print("• Use Databricks Runtime ML with pre-installed packages")

# Store results for next cells
print(f"\n✅ Data preparation complete!")
print(f"📊 Final training data shape: {X_train_smote.shape}")
print(f"🏷️ Final training labels shape: {y_train_smote.shape}")

In [0]:
# Method 2: Spark-based Oversampling (Scalable Approach)
print("🔥 Method 2: Spark Oversampling - Scalable to Big Data")

from functools import reduce
from pyspark.ml.feature import VectorAssembler

def oversample_minority_classes(df, label_col):
    """
    Oversample minority classes using Spark's distributed processing.
    This method scales to millions of records efficiently.
    """
    from pyspark.sql.functions import col as pyspark_col  # Explicitly import within scope
    
    # Get class counts
    class_counts = df.groupBy(label_col).count().collect()
    max_count = max([row['count'] for row in class_counts])
    
    print(f"📊 Maximum class count: {max_count}")
    print(f"🏷️ Class distribution before oversampling:")
    for row in class_counts:
        print(f"   Class {row[label_col]}: {row['count']} samples")
    
    oversampled_dfs = []
    
    for row in class_counts:
        cls = row[label_col]
        count = row['count']
        
        if count < max_count:
            # Calculate how many times to duplicate
            ratio = int(max_count / count)
            remainder = max_count % count
            
            print(f"🔄 Class {cls}: duplicating {ratio-1} times + {remainder} extra samples")
            
            # Get class-specific data
            df_cls = df.filter(pyspark_col(label_col) == cls)
            
            # Create oversampled version
            oversampled = df_cls
            
            # Add duplicates (ratio-1 times)
            for _ in range(ratio - 1):
                oversampled = oversampled.union(df_cls)
            
            # Add remainder samples
            if remainder > 0:
                oversampled = oversampled.union(df_cls.limit(remainder))
            
            oversampled_dfs.append(oversampled)
        else:
            # Majority class - add as-is
            oversampled_dfs.append(df.filter(pyspark_col(label_col) == cls))
    
    # Union all oversampled classes
    balanced_df = reduce(lambda a, b: a.union(b), oversampled_dfs)
    
    # Shuffle the data
    balanced_df = balanced_df.orderBy(pyspark_col("__INDEX"))
    
    return balanced_df

# Add index column for shuffling
train_df_spark_indexed = train_df_spark.withColumn("__INDEX", expr("rand()"))

# Apply Spark oversampling
print("\n� Applying Spark-based oversampling...")
train_df_spark_balanced = oversample_minority_classes(train_df_spark_indexed, "CLASS")

# Remove index column
train_df_spark_balanced = train_df_spark_balanced.drop("__INDEX")

print(f"\n� Spark Oversampling Results:")
print(f"🏋️ Original training size: {train_df_spark.count():,}")
print(f"� Balanced training size: {train_df_spark_balanced.count():,}")

# Display final class distribution
print("\n🏷️ Class Distribution - After Spark Oversampling:")
balanced_counts = train_df_spark_balanced.groupBy("CLASS").count().orderBy("CLASS").collect()
for row in balanced_counts:
    print(f"   Class {row['CLASS']}: {row['count']} samples")

# Convert to pandas for comparison
train_spark_balanced_pandas = train_df_spark_balanced.toPandas()
X_train_spark_balanced = train_spark_balanced_pandas[feature_columns]
y_train_spark_balanced = train_spark_balanced_pandas['CLASS']

print("\n💡 Spark Oversampling Benefits:")
print("• Scales to millions of records efficiently")
print("• Simple and reliable implementation")
print("• Fast processing with distributed computing")
print("• No external dependencies required")
print("• Works with any dataset size")

In [0]:
# Optimal Method Selection for Wine Dataset
print("🎯 Optimal Method Selection for Wine Dataset")

# Analyze the specific characteristics of our wine dataset
print("📊 Wine Dataset Analysis:")
print(f"• Dataset size: {len(X_train):,} samples")
print(f"• Number of features: {len(feature_columns)}")
print(f"• Number of classes: {len(original_counts)}")
print(f"• Feature types: All numeric, continuous")
print(f"• Domain: Wine classification (chemical properties)")

# Class imbalance analysis
print(f"\n🏷️ Class Imbalance Analysis:")
total_samples = len(X_train)
for cls, count in original_counts.items():
    percentage = (count / total_samples) * 100
    print(f"• Class {cls}: {count} samples ({percentage:.1f}%)")

# Calculate imbalance ratio
max_count = max(original_counts.values)
min_count = min(original_counts.values)
imbalance_ratio = max_count / min_count

print(f"\n📈 Imbalance Ratio: {imbalance_ratio:.2f}:1")

# Decision based on specific wine dataset characteristics
print(f"\n🔍 Decision Factors for Wine Dataset:")
print(f"✅ Small dataset (< 1K samples) - SMOTE excels")
print(f"✅ Moderate imbalance (2.7:1 ratio) - Both methods work")
print(f"✅ All numeric features - Perfect for SMOTE interpolation")
print(f"✅ Chemical properties - Meaningful feature relationships")
print(f"✅ No high-cardinality categorical variables - SMOTE friendly")
print(f"✅ Quality-critical domain - SMOTE's synthetic quality preferred")

# Final recommendation
print(f"\n� FINAL RECOMMENDATION: SMOTE")
print(f"=" * 50)
print(f"💡 Why SMOTE is optimal for this wine dataset:")
print(f"")
print(f"1. 🎯 Dataset Size Perfect Match:")
print(f"   • {len(X_train)} samples - ideal for SMOTE's computational efficiency")
print(f"   • Small enough for high-quality synthetic sample generation")
print(f"")
print(f"2. 🧪 Domain-Specific Advantages:")
print(f"   • Wine chemical properties have continuous, meaningful relationships")
print(f"   • SMOTE preserves these relationships when creating synthetic samples")
print(f"   • Chemical feature interpolation is scientifically meaningful")
print(f"")
print(f"3. 📊 Quality Over Quantity:")
print(f"   • SMOTE creates diverse synthetic samples vs simple duplicates")
print(f"   • Better generalization to new wine varieties")
print(f"   • Reduces overfitting risk compared to oversampling")
print(f"")
print(f"4. ⚖️ Imbalance Level:")
print(f"   • {imbalance_ratio:.1f}:1 ratio is moderate - perfect for SMOTE")
print(f"   • Not severe enough to require massive duplication")
print(f"   • SMOTE handles this level efficiently")

# Apply SMOTE as the optimal choice
print(f"\n🚀 Applying SMOTE (Optimal Choice for Wine Dataset)...")

# Apply SMOTE
smote = SMOTE(random_state=42, k_neighbors=min(3, min(original_counts.values)-1))
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"\n✅ SMOTE Applied Successfully!")
print(f"📊 Original training size: {len(X_train):,}")
print(f"🧬 SMOTE balanced size: {len(X_train_balanced):,}")

# Display final class distribution
print(f"\n🏷️ Final Class Distribution (SMOTE):")
balanced_counts = pd.Series(y_train_balanced).value_counts().sort_index()
for cls, count in balanced_counts.items():
    original_count = original_counts[cls]
    increase = count - original_count
    print(f"• Class {cls}: {count} samples (+{increase} synthetic)")

# Visualize the SMOTE result
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Original distribution
axes[0].bar(original_counts.index, original_counts.values, color='lightcoral', alpha=0.7)
axes[0].set_title(f'Original Distribution\n{len(X_train)} samples', fontweight='bold')
axes[0].set_xlabel('Wine Class')
axes[0].set_ylabel('Number of Samples')
axes[0].grid(True, alpha=0.3)

# SMOTE balanced distribution
axes[1].bar(balanced_counts.index, balanced_counts.values, color='lightblue', alpha=0.7)
axes[1].set_title(f'SMOTE Balanced\n{len(X_train_balanced)} samples', fontweight='bold')
axes[1].set_xlabel('Wine Class')
axes[1].set_ylabel('Number of Samples')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n� SMOTE - Optimal Choice Confirmed!")
print(f"• Perfect for small, numeric wine datasets")
print(f"• Preserves chemical property relationships")
print(f"• Creates high-quality synthetic wine samples")
print(f"• Ready for sklearn model training")

## 🤖 **Model Training with sklearn - The Serving Advantage**

### 🎯 **Objective**
Train machine learning models using sklearn for optimal serving capabilities.

### 🤖 **Why sklearn for Models?**

| **Aspect** | **sklearn** | **Spark MLlib** |
|------------|-------------|-----------------|
| Model Serving | ✅ **Easy & Reliable** | ⚠️ Complex setup |
| Model Size | ✅ **Small** | ❌ Large (Spark overhead) |
| Inference Speed | ✅ **Fast** | ⚠️ Slower (Spark initialization) |
| Documentation | ✅ **Extensive** | ⚠️ Limited |
| Community Support | ✅ **Huge** | ⚠️ Smaller |
| Production Readiness | ✅ **Battle-tested** | ⚠️ Complex deployment |

### 🔄 **Data Conversion Strategy**
1. **Spark** → **pandas**: Convert processed data for sklearn
2. **sklearn**: Train models efficiently
3. **MLflow**: Log and version models
4. **Unity Catalog**: Register for production use

In [0]:
# Convert Spark DataFrames to pandas for sklearn training
print("🔄 Converting Spark DataFrames to pandas for sklearn training...")

# Convert training data
train_df_pandas = train_df_spark.toPandas()
test_df_pandas = test_df_spark.toPandas()

# Separate features and target
feature_columns = [col for col in train_df_pandas.columns if col != 'CLASS']

X_train = train_df_pandas[feature_columns]
y_train = train_df_pandas['CLASS']

X_test = test_df_pandas[feature_columns]
y_test = test_df_pandas['CLASS']

print(f"📊 Training data shape: {X_train.shape}")
print(f"📊 Test data shape: {X_test.shape}")
print(f"🏷️ Number of features: {len(feature_columns)}")

# Display feature names
print("\n📋 Feature List:")
for i, feature in enumerate(feature_columns, 1):
    print(f"{i:2d}. {feature}")

In [0]:
# Feature Scaling with sklearn (SMOTE-Balanced Dataset)
print("⚖️ Feature Scaling with sklearn (SMOTE-Balanced Dataset)...")

# Initialize scaler
sklearn_scaler = SklearnStandardScaler()

# Fit on SMOTE-balanced training data and transform both sets
X_train_balanced_scaled = sklearn_scaler.fit_transform(X_train_balanced)
X_test_scaled = sklearn_scaler.transform(X_test)

# Convert back to DataFrame for easier handling
X_train_balanced_scaled_df = pd.DataFrame(X_train_balanced_scaled, columns=feature_columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_columns)

print("\n✅ Scaling complete!")
print(f"📊 SMOTE-balanced training data shape: {X_train_balanced_scaled_df.shape}")
print(f"📊 Test data shape: {X_test_scaled_df.shape}")

# Show scaling statistics
print("\n📈 Scaling Statistics (SMOTE-Balanced Training Data):")
print(f"📏 Mean (should be ~0): {np.mean(X_train_balanced_scaled, axis=0)[:5]}...")
print(f"📏 Std (should be ~1): {np.std(X_train_balanced_scaled, axis=0)[:5]}...")

# Visualize scaling effect with SMOTE-balanced data
plt.figure(figsize=(12, 6))

# Convert to float to avoid Decimal type issues
X_train_balanced_for_plot = X_train_balanced.astype(float).iloc[:, :5]
X_train_balanced_scaled_for_plot = pd.DataFrame(X_train_balanced_scaled, columns=feature_columns).astype(float).iloc[:, :5]

plt.subplot(1, 2, 1)
plt.boxplot(X_train_balanced_for_plot.values)
plt.title('Before Scaling\n(SMOTE-Balanced - First 5 Features)', fontweight='bold')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.boxplot(X_train_balanced_scaled_for_plot.values)
plt.title('After Scaling\n(SMOTE-Balanced - First 5 Features)', fontweight='bold')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print("\n💡 SMOTE + Scaling Benefits:")
print("• SMOTE ensures equal wine class representation")
print("• Scaling prevents chemical property dominance")
print("• Combined: Better model performance on all wine types")
print("• Optimal for wine classification accuracy")

In [0]:
# Model Training with MLflow Tracking (SMOTE-Optimized)
print("🤖 Training ALL Available Classification Models with MLflow Tracking...")

# Set up MLflow experiment
mlflow.set_experiment("/Users/viniciusdvale@gmail.com/wine_production_hybrid")



# Define ALL available models with appropriate configurations
models = {}

# Linear Models
models["Logistic Regression"] = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
models["Ridge Classifier"] = RidgeClassifier(random_state=42)
models["SGD Classifier"] = SGDClassifier(random_state=42, max_iter=1000)
models["Perceptron"] = Perceptron(random_state=42, max_iter=1000)
models["Passive Aggressive"] = PassiveAggressiveClassifier(random_state=42, max_iter=1000)

# Tree-based Models
models["Decision Tree"] = DecisionTreeClassifier(random_state=42, max_depth=8)
models["Random Forest"] = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
models["Gradient Boosting"] = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=6, learning_rate=0.1)
models["AdaBoost"] = AdaBoostClassifier(n_estimators=100, random_state=42, learning_rate=0.1)
models["Extra Trees"] = ExtraTreesClassifier(n_estimators=100, random_state=42, max_depth=10)
models["Bagging"] = BaggingClassifier(n_estimators=100, random_state=42)

# Support Vector Machines
models["SVM (RBF)"] = SVC(random_state=42, probability=True)
models["Linear SVM"] = LinearSVC(random_state=42, max_iter=1000)
models["Nu SVM"] = NuSVC(random_state=42, probability=True)

# Nearest Neighbors
models["KNN"] = KNeighborsClassifier(n_neighbors=5, weights='uniform')

# Naive Bayes
models["Gaussian NB"] = GaussianNB()
models["Multinomial NB"] = MultinomialNB()
models["Bernoulli NB"] = BernoulliNB()

# Discriminant Analysis
models["Linear Discriminant"] = LinearDiscriminantAnalysis()
models["Quadratic Discriminant"] = QuadraticDiscriminantAnalysis()

# Neural Network
models["MLP Classifier"] = MLPClassifier(random_state=42, max_iter=1000, hidden_layer_sizes=(100,))

# Add Gaussian Process
models["Gaussian Process"] = GaussianProcessClassifier(random_state=42, kernel=1.0 * RBF(1.0))

# Add semi-supervised models
# Note: These need unlabeled data, but we'll try with our data
models["Label Propagation"] = LabelPropagation()
models["Label Spreading"] = LabelSpreading()


# Ensemble combinations
# Voting Classifier (soft voting with top models)
voting_models = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ('lr', LogisticRegression(random_state=42, max_iter=500))
]
models["Voting (Soft)"] = VotingClassifier(estimators=voting_models, voting='soft')

# Stacking Classifier
stacking_models = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ('svm', SVC(random_state=42, probability=True))
]
models["Stacking"] = StackingClassifier(
    estimators=stacking_models, 
    final_estimator=LogisticRegression(random_state=42, max_iter=500),
    cv=3
)

print(f"✅ Loaded {len(models)} classification models!")
print(f"📊 Model categories:")
print(f"   • Linear: {len([m for m in models.keys() if any(x in m.lower() for x in ['logistic', 'ridge', 'sgd', 'perceptron', 'passive'])])}")
print(f"   • Tree-based: {len([m for m in models.keys() if any(x in m.lower() for x in ['tree', 'forest', 'boosting', 'adaboost', 'extra', 'bagging'])])}")
print(f"   • SVM: {len([m for m in models.keys() if 'svm' in m.lower()])}")
print(f"   • Bayesian: {len([m for m in models.keys() if 'nb' in m.lower() or 'discriminant' in m.lower()])}")
print(f"   • Neural: {len([m for m in models.keys() if 'mlp' in m.lower()])}")
print(f"   • Ensemble: {len([m for m in models.keys() if any(x in m.lower() for x in ['voting', 'stacking'])])}")

# Training results storage
training_results = []
trained_models = {}

print(f"\n🏋️ Training ALL {len(models)} models...")

for model_name, model in models.items():
    print(f"\n🤖 Training {model_name}...")
    
    with mlflow.start_run(run_name=f"{model_name}_SMOTE_Hybrid") as run:
        try:
            # Train model on SMOTE-balanced data
            model.fit(X_train_balanced_scaled, y_train_balanced)
            
            # Make predictions on test set
            y_pred = model.predict(X_test_scaled)
            
            # Calculate probabilities if available
            if hasattr(model, 'predict_proba'):
                y_proba = model.predict_proba(X_test_scaled)
                has_probabilities = True
            else:
                y_proba = None
                has_probabilities = False
            
            # Calculate comprehensive metrics
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
            recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
            f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
            
            # Additional metrics
            from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef
            balanced_acc = balanced_accuracy_score(y_test, y_pred)
            mcc = matthews_corrcoef(y_test, y_pred)
            
            # Log parameters and metrics
            mlflow.log_param("model_type", model_name)
            mlflow.log_param("balancing_method", "SMOTE")
            mlflow.log_param("original_train_size", len(X_train))
            mlflow.log_param("smote_balanced_size", len(X_train_balanced))
            mlflow.log_param("has_probabilities", has_probabilities)
            
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("balanced_accuracy", balanced_acc)
            mlflow.log_metric("matthews_corrcoef", mcc)
            mlflow.log_metric("test_samples", len(X_test))
            mlflow.log_metric("n_features", len(feature_columns))
            
            # Log SMOTE-specific metrics
            mlflow.log_metric("synthetic_samples_created", len(X_train_balanced) - len(X_train))
            mlflow.log_metric("imbalance_ratio_before", imbalance_ratio)
            mlflow.log_metric("class_balance_achieved", 1.0)
            
            # Log class distribution
            for cls in sorted(set(y_train_balanced)):
                train_count = sum(y_train_balanced == cls)
                test_count = sum(y_test == cls)
                mlflow.log_metric(f"train_class_{cls}_count", train_count)
                mlflow.log_metric(f"test_class_{cls}_count", test_count)
            
            # Log model with signature and input example
            signature = infer_signature(X_train_balanced_scaled, y_pred)
            input_example = X_train_balanced_scaled[:5]
            
            mlflow.sklearn.log_model(
                model, 
                name="model", 
                signature=signature,
                input_example=input_example
            )
            
            # Store results
            training_results.append({
                "model_name": model_name,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1_score": f1,
                "balanced_accuracy": balanced_acc,
                "matthews_corrcoef": mcc,
                "has_probabilities": has_probabilities,
                "run_id": run.info.run_id,
                "balancing_method": "SMOTE",
                "train_size": len(X_train_balanced)
            })
            
            trained_models[model_name] = model
            
            print(f"✅ {model_name} - Acc: {accuracy:.4f}, F1: {f1:.4f}, MCC: {mcc:.4f}")
            
        except Exception as e:
            print(f"❌ {model_name} failed: {str(e)}")
            # Log the error to MLflow
            mlflow.log_param("error", str(e))
            continue

# Create comprehensive results DataFrame
results_df = pd.DataFrame(training_results)

if len(results_df) > 0:
    # Sort by accuracy
    results_sorted = results_df.sort_values('accuracy', ascending=False)
    
    print(f"\n📊 COMPREHENSIVE Model Comparison Results ({len(results_df)} models trained):")
    print("=" * 100)
    
    # Display top performers
    print("\n🏆 TOP 10 MODELS BY ACCURACY:")
    top_10 = results_sorted.head(10)
    for i, (_, row) in enumerate(top_10.iterrows(), 1):
        print(f"{i:2d}. {row['model_name']:<25} | Acc: {row['accuracy']:.4f} | F1: {row['f1_score']:.4f} | MCC: {row['matthews_corrcoef']:.4f}")
    
    print(f"\n📈 MODEL CATEGORY PERFORMANCE:")
    categories = {
        'Linear': ['logistic', 'ridge', 'sgd', 'perceptron', 'passive'],
        'Tree': ['tree', 'forest', 'boosting', 'adaboost', 'extra', 'bagging'],
        'SVM': ['svm'],
        'Bayesian': ['nb', 'discriminant'],
        'Neural': ['mlp'],
        'Ensemble': ['voting', 'stacking']
    }
    
    for cat, keywords in categories.items():
        cat_models = results_sorted[results_sorted['model_name'].str.contains('|'.join(keywords), case=False, na=False)]
        if len(cat_models) > 0:
            best_cat = cat_models.iloc[0]
            print(f"• {cat:<10}: {best_cat['model_name']:<25} (Acc: {best_cat['accuracy']:.4f})")
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('🤖 Comprehensive Model Comparison - ALL Classification Algorithms', fontsize=16, fontweight='bold')
    
    # Top 15 models by accuracy
    top_15 = results_sorted.head(15)
    
    # Accuracy comparison
    axes[0, 0].barh(range(len(top_15)), top_15['accuracy'], color='skyblue', alpha=0.8)
    axes[0, 0].set_yticks(range(len(top_15)))
    axes[0, 0].set_yticklabels([name.split()[0] for name in top_15['model_name']], fontsize=8)
    axes[0, 0].set_xlabel('Accuracy')
    axes[0, 0].set_title('🎯 Top 15 Models - Accuracy', fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    
    # F1 Score comparison
    axes[0, 1].barh(range(len(top_15)), top_15['f1_score'], color='lightgreen', alpha=0.8)
    axes[0, 1].set_yticks(range(len(top_15)))
    axes[0, 1].set_yticklabels([name.split()[0] for name in top_15['model_name']], fontsize=8)
    axes[0, 1].set_xlabel('F1 Score')
    axes[0, 1].set_title('📈 Top 15 Models - F1 Score', fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Matthews Correlation Coefficient
    axes[0, 2].barh(range(len(top_15)), top_15['matthews_corrcoef'], color='orange', alpha=0.8)
    axes[0, 2].set_yticks(range(len(top_15)))
    axes[0, 2].set_yticklabels([name.split()[0] for name in top_15['model_name']], fontsize=8)
    axes[0, 2].set_xlabel('Matthews Correlation Coefficient')
    axes[0, 2].set_title('🎲 Top 15 Models - MCC', fontweight='bold')
    axes[0, 2].grid(True, alpha=0.3)
    
    # Accuracy vs F1 scatter
    axes[1, 0].scatter(results_sorted['accuracy'], results_sorted['f1_score'], 
                       s=60, alpha=0.7, c=range(len(results_sorted)), cmap='viridis')
    axes[1, 0].set_xlabel('Accuracy')
    axes[1, 0].set_ylabel('F1 Score')
    axes[1, 0].set_title('🎯 Accuracy vs F1 Score', fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Category performance box plot
    category_data = []
    category_labels = []
    for cat, keywords in categories.items():
        cat_models = results_sorted[results_sorted['model_name'].str.contains('|'.join(keywords), case=False, na=False)]
        if len(cat_models) > 0:
            category_data.append(cat_models['accuracy'])
            category_labels.append(cat)
    
    if category_data:
        axes[1, 1].boxplot(category_data, labels=category_labels, patch_artist=True)
        axes[1, 1].set_ylabel('Accuracy')
        axes[1, 1].set_title('📊 Performance by Model Category', fontweight='bold')
        axes[1, 1].tick_params(axis='x', rotation=45)
        axes[1, 1].grid(True, alpha=0.3)
    
    # Model complexity vs performance (simplified)
    complexity_scores = []
    for name in results_sorted['model_name']:
        if any(x in name.lower() for x in ['logistic', 'ridge', 'sgd', 'perceptron', 'passive']):
            complexity_scores.append(1)  # Low complexity
        elif any(x in name.lower() for x in ['tree', 'nb', 'discriminant']):
            complexity_scores.append(2)  # Medium complexity
        elif any(x in name.lower() for x in ['svm', 'knn']):
            complexity_scores.append(3)  # High complexity
        elif any(x in name.lower() for x in ['forest', 'boosting', 'adaboost', 'extra', 'bagging']):
            complexity_scores.append(4)  # Very high complexity
        else:
            complexity_scores.append(5)  # Extreme complexity
    
    axes[1, 2].scatter(complexity_scores, results_sorted['accuracy'], 
                       s=60, alpha=0.7, c='red')
    axes[1, 2].set_xlabel('Model Complexity (1=Low, 5=Extreme)')
    axes[1, 2].set_ylabel('Accuracy')
    axes[1, 2].set_title('⚖️ Complexity vs Performance', fontweight='bold')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Final summary
    best_model_name = results_sorted.iloc[0]['model_name']
    best_accuracy = results_sorted.iloc[0]['accuracy']
    best_f1 = results_sorted.iloc[0]['f1_score']
    
    print(f"\n🏆 OVERALL BEST MODEL: {best_model_name}")
    print(f"📊 Accuracy: {best_accuracy:.4f}")
    print(f"📈 F1 Score: {best_f1:.4f}")
    print(f"🧪 Method: SMOTE-Optimized Hybrid Pipeline")
    print(f"📊 Training Samples: {len(X_train_balanced):,}")
    
    print(f"\n💡 COMPREHENSIVE INSIGHTS:")
    print(f"• Total models trained: {len(results_df)}")
    print(f"• Best accuracy achieved: {best_accuracy:.4f}")
    print(f"• Models with probabilities: {sum(results_df['has_probabilities'])}/{len(results_df)}")
    print(f"• Average accuracy across all models: {results_df['accuracy'].mean():.4f}")
    print(f"• Accuracy range: {results_df['accuracy'].min():.4f} - {results_df['accuracy'].max():.4f}")
    
    # Store best model for next steps
    best_model = trained_models[best_model_name]
    
else:
    print("❌ No models were successfully trained!")
    best_model = None

print(f"\n✅ Comprehensive model training complete!")

## 🔧 **Hyperparameter Tuning & Fine-Tuning - Model Optimization**

### 🎯 **Objective**
Optimize the best performing model using comprehensive hyperparameter tuning.

### 🔥 **Why Hyperparameter Tuning Matters**

| **Aspect** | **Default Parameters** | **Tuned Parameters** |
|------------|-------------------------|----------------------|
| **Performance** | Good baseline | **Optimized performance** |
| **Generalization** | May overfit/underfit | **Better generalization** |
| **Efficiency** | Suboptimal resource use | **Optimized resource use** |
| **Production Ready** | Not production optimized | **Production-grade model** |

### 📋 **Tuning Strategy**
1. **Grid Search**: Exhaustive parameter search
2. **Cross-Validation**: Robust performance evaluation
3. **Multiple Metrics**: Comprehensive optimization
4. **MLflow Tracking**: Complete experiment logging

In [0]:
# Hyperparameter Tuning with Grid Search and Cross-Validation
print("🔧 Starting Hyperparameter Tuning for Best Model...")

# Get the best model from previous comprehensive comparison
if len(results_df) > 0:
    best_model_name = results_sorted.iloc[0]['model_name']
    print(f"� Best model from comparison: {best_model_name}")
    
    # Define parameter grids for different model types
    param_grids = {}
    
    # Random Forest parameters
    if 'Random Forest' in best_model_name:
        param_grids['Random Forest'] = {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, 15, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None]
        }
        base_model = RandomForestClassifier(random_state=42)
    
    # Gradient Boosting parameters
    elif 'Gradient Boosting' in best_model_name:
        param_grids['Gradient Boosting'] = {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.1, 0.2],
            'subsample': [0.8, 0.9, 1.0],
            'min_samples_split': [2, 5, 10]
        }
        base_model = GradientBoostingClassifier(random_state=42)
    
    # Logistic Regression parameters
    elif 'Logistic Regression' in best_model_name:
        param_grids['Logistic Regression'] = {
            'C': [0.1, 1.0, 10.0, 100.0],
            'solver': ['lbfgs', 'liblinear', 'newton-cg'],
            'max_iter': [500, 1000, 2000],
            'penalty': ['l2', 'none']
        }
        base_model = LogisticRegression(random_state=42)
    
    # SVM parameters
    elif 'SVM' in best_model_name:
        param_grids['SVM'] = {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear'],
            'gamma': ['scale', 'auto', 0.001, 0.01],
            'probability': [True]
        }
        base_model = SVC(random_state=42)
    
    # Decision Tree parameters
    elif 'Decision Tree' in best_model_name:
        param_grids['Decision Tree'] = {
            'max_depth': [3, 5, 7, 10, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'criterion': ['gini', 'entropy'],
            'max_features': ['sqrt', 'log2', None]
        }
        base_model = DecisionTreeClassifier(random_state=42)
    
    # KNN parameters
    elif 'KNN' in best_model_name:
        param_grids['KNN'] = {
            'n_neighbors': [3, 5, 7, 9],
            'weights': ['uniform', 'distance'],
            'algorithm': ['auto', 'ball_tree', 'kd_tree'],
            'p': [1, 2]  # Manhattan vs Euclidean distance
        }
        base_model = KNeighborsClassifier()
    
    # AdaBoost parameters
    elif 'AdaBoost' in best_model_name:
        param_grids['AdaBoost'] = {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.1, 0.5, 1.0],
            'algorithm': ['SAMME', 'SAMME.R']
        }
        base_model = AdaBoostClassifier(random_state=42)
    
    # Extra Trees parameters
    elif 'Extra Trees' in best_model_name:
        param_grids['Extra Trees'] = {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, 15, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None]
        }
        base_model = ExtraTreesClassifier(random_state=42)
    
    # MLP Classifier parameters
    elif 'MLP' in best_model_name:
        param_grids['MLP'] = {
            'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
            'activation': ['relu', 'tanh', 'logistic'],
            'solver': ['adam', 'sgd'],
            'alpha': [0.0001, 0.001, 0.01],
            'learning_rate': ['constant', 'adaptive'],
            'max_iter': [500, 1000]
        }
        base_model = MLPClassifier(random_state=42)
    
    # Default fallback for other models
    else:
        print(f"⚠️ No specific parameter grid defined for {best_model_name}")
        print("🔄 Using a generic parameter grid...")
        param_grids['Generic'] = {
            'random_state': [42]
        }
        base_model = trained_models[best_model_name]
    
    # Get the appropriate parameter grid
    model_type = None
    for key in param_grids.keys():
        if key in best_model_name:
            model_type = key
            break
    
    if model_type is None:
        model_type = list(param_grids.keys())[0]
    
    param_grid = param_grids[model_type]
    
    print(f"\n📊 Parameter Grid for {model_type}:")
    for param, values in param_grid.items():
        print(f"   {param}: {values}")
    
    print(f"\n🔧 Starting Grid Search with {len(param_grid)} parameters...")
    
    # Perform Grid Search with Cross-Validation
    with mlflow.start_run(run_name=f"Hyperparameter_Tuning_{best_model_name}") as run:
        # Set up GridSearchCV
        grid_search = GridSearchCV(
            estimator=base_model,
            param_grid=param_grid,
            cv=5,  # 5-fold cross-validation
            scoring='accuracy',
            n_jobs=-1,  # Use all available cores
            verbose=1,
            return_train_score=True
        )
        
        print("🏋️ Training Grid Search...")
        grid_search.fit(X_train_balanced_scaled, y_train_balanced)
        
        # Get best parameters and model
        best_params = grid_search.best_params_
        best_cv_score = grid_search.best_score_
        best_tuned_model = grid_search.best_estimator_
        
        # Make predictions on test set
        y_pred_tuned = best_tuned_model.predict(X_test_scaled)
        
        # Calculate comprehensive metrics
        accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
        precision_tuned = precision_score(y_test, y_pred_tuned, average='weighted', zero_division=0)
        recall_tuned = recall_score(y_test, y_pred_tuned, average='weighted', zero_division=0)
        f1_tuned = f1_score(y_test, y_pred_tuned, average='weighted', zero_division=0)
        
        print(f"\n🎯 Hyperparameter Tuning Results:")
        print(f"✅ Best CV Score: {best_cv_score:.4f}")
        print(f"📊 Test Accuracy: {accuracy_tuned:.4f}")
        print(f"📈 Test F1 Score: {f1_tuned:.4f}")
        print(f"🔧 Best Parameters:")
        for param, value in best_params.items():
            print(f"   {param}: {value}")
        
        # Log all results to MLflow
        mlflow.log_param("model_type", f"{best_model_name}_Tuned")
        mlflow.log_param("tuning_method", "GridSearchCV")
        mlflow.log_param("cv_folds", 5)
        mlflow.log_param("total_combinations", len(list(grid_search.cv_results_['params'])))
        
        # Log best parameters
        for param, value in best_params.items():
            mlflow.log_param(f"best_{param}", value)
        
        # Log metrics
        mlflow.log_metric("best_cv_score", best_cv_score)
        mlflow.log_metric("test_accuracy", accuracy_tuned)
        mlflow.log_metric("test_precision", precision_tuned)
        mlflow.log_metric("test_recall", recall_tuned)
        mlflow.log_metric("test_f1_score", f1_tuned)
        
        # Log improvement over baseline
        baseline_accuracy = results_sorted.iloc[0]['accuracy']
        improvement = accuracy_tuned - baseline_accuracy
        mlflow.log_metric("accuracy_improvement", improvement)
        mlflow.log_metric("baseline_accuracy", baseline_accuracy)
        
        # Log the tuned model
        signature = infer_signature(X_train_balanced_scaled, y_pred_tuned)
        mlflow.sklearn.log_model(
            best_tuned_model,
            "tuned_model",
            signature=signature,
            input_example=X_train_balanced_scaled[:5]
        )
        
        print(f"\n📈 Performance Improvement:")
        print(f"📊 Baseline Accuracy: {baseline_accuracy:.4f}")
        print(f"🎯 Tuned Accuracy: {accuracy_tuned:.4f}")
        print(f"📈 Improvement: {improvement:+.4f} ({(improvement/baseline_accuracy)*100:+.2f}%)")
        
        # Create visualization of tuning results
        plt.figure(figsize=(15, 10))
        
        # Convert cv_results to DataFrame for easier plotting
        cv_results_df = pd.DataFrame(grid_search.cv_results_)
        
        # Plot 1: Mean test scores vs parameter combinations
        plt.subplot(2, 3, 1)
        plt.plot(range(len(cv_results_df)), cv_results_df['mean_test_score'], 'b-', alpha=0.7)
        plt.axhline(y=baseline_accuracy, color='r', linestyle='--', label=f'Baseline: {baseline_accuracy:.4f}')
        plt.axhline(y=accuracy_tuned, color='g', linestyle='-', label=f'Tuned: {accuracy_tuned:.4f}')
        plt.xlabel('Parameter Combination Index')
        plt.ylabel('Mean CV Score')
        plt.title('Grid Search Progress', fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Plot 2: Train vs Test scores
        plt.subplot(2, 3, 2)
        plt.scatter(cv_results_df['mean_train_score'], cv_results_df['mean_test_score'], alpha=0.6)
        plt.plot([cv_results_df['mean_train_score'].min(), cv_results_df['mean_train_score'].max()],
                [cv_results_df['mean_train_score'].min(), cv_results_df['mean_train_score'].max()],
                'r--', alpha=0.8)
        plt.xlabel('Mean Train Score')
        plt.ylabel('Mean Test Score')
        plt.title('Train vs Test Scores', fontweight='bold')
        plt.grid(True, alpha=0.3)
        
        # Plot 3: Top 10 parameter combinations
        top_10_combinations = cv_results_df.nlargest(10, 'mean_test_score')
        plt.subplot(2, 3, 3)
        plt.barh(range(10), top_10_combinations['mean_test_score'], color='skyblue', alpha=0.8)
        plt.yticks(range(10), [f"Combo {i+1}" for i in range(10)])
        plt.xlabel('Mean CV Score')
        plt.title('Top 10 Parameter Combinations', fontweight='bold')
        plt.grid(True, alpha=0.3)
        
        # Plot 4: Parameter importance (if applicable)
        if len(best_params) > 1:
            plt.subplot(2, 3, 4)
            param_importance = {}
            for param in param_grid.keys():
                if param in best_params:
                    # Simple importance based on parameter variation impact
                    param_values = cv_results_df[f'param_{param}'].unique()
                    scores_by_param = []
                    for val in param_values:
                        mask = cv_results_df[f'param_{param}'] == val
                        scores_by_param.append(cv_results_df[mask]['mean_test_score'].mean())
                    
                    if len(scores_by_param) > 1:
                        importance = max(scores_by_param) - min(scores_by_param)
                        param_importance[param] = importance
            
            if param_importance:
                params = list(param_importance.keys())
                importance_values = list(param_importance.values())
                plt.barh(params, importance_values, color='orange', alpha=0.8)
                plt.xlabel('Score Range')
                plt.title('Parameter Importance', fontweight='bold')
                plt.grid(True, alpha=0.3)
        
        # Plot 5: Fit time vs performance
        plt.subplot(2, 3, 5)
        plt.scatter(cv_results_df['mean_fit_time'], cv_results_df['mean_test_score'], alpha=0.6)
        plt.xlabel('Mean Fit Time (seconds)')
        plt.ylabel('Mean Test Score')
        plt.title('Performance vs Training Time', fontweight='bold')
        plt.grid(True, alpha=0.3)
        
        # Plot 6: Comparison with baseline
        plt.subplot(2, 3, 6)
        models_compare = ['Baseline', 'Tuned']
        accuracies_compare = [baseline_accuracy, accuracy_tuned]
        colors = ['lightcoral', 'lightgreen']
        bars = plt.bar(models_compare, accuracies_compare, color=colors, alpha=0.8)
        plt.ylabel('Accuracy')
        plt.title('Baseline vs Tuned Model', fontweight='bold')
        plt.grid(True, alpha=0.3)
        
        # Add value labels
        for bar, acc in zip(bars, accuracies_compare):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        # Store the tuned model as the new best model
        tuned_model = best_tuned_model
        tuned_accuracy = accuracy_tuned
        tuned_params = best_params
        
        print(f"\n✅ Hyperparameter tuning complete!")
        print(f"🏆 Final tuned model: {best_model_name} with optimized parameters")
        print(f"📊 Final accuracy: {tuned_accuracy:.4f}")
        
else:
    print("❌ No models available for tuning!")
    tuned_model = None
    tuned_accuracy = None
    tuned_params = None

## 📈 **Model Evaluation and Selection**

### 🎯 **Objective**
Comprehensive evaluation of tuned model vs baseline models with detailed analysis.

### 📊 **Evaluation Framework**
1. **Baseline vs Tuned**: Direct performance comparison
2. **Multiple Metrics**: Accuracy, Precision, Recall, F1, MCC
3. **Statistical Analysis**: Confidence intervals and significance
4. **Production Readiness**: Model complexity and deployment considerations

In [0]:
# Detailed Model Evaluation
print("📈 Detailed Model Evaluation...")

# Compare baseline vs tuned models
if tuned_model is not None and len(results_df) > 0:
    
    # Get baseline best model
    baseline_model_name = results_sorted.iloc[0]['model_name']
    baseline_model = trained_models[baseline_model_name]
    baseline_accuracy = results_sorted.iloc[0]['accuracy']
    
    print(f"🏆 Baseline Model: {baseline_model_name}")
    print(f"📊 Baseline Accuracy: {baseline_accuracy:.4f}")
    print(f"🎯 Tuned Accuracy: {tuned_accuracy:.4f}")
    print(f"📈 Improvement: {(tuned_accuracy - baseline_accuracy):+.4f}")
    
    # Make predictions with both models for detailed comparison
    y_pred_baseline = baseline_model.predict(X_test_scaled)
    y_pred_tuned = tuned_model.predict(X_test_scaled)
    
    # Calculate comprehensive metrics for both models
    from sklearn.metrics import classification_report, confusion_matrix, matthews_corrcoef
    
    # Baseline metrics
    baseline_precision = precision_score(y_test, y_pred_baseline, average='weighted', zero_division=0)
    baseline_recall = recall_score(y_test, y_pred_baseline, average='weighted', zero_division=0)
    baseline_f1 = f1_score(y_test, y_pred_baseline, average='weighted', zero_division=0)
    baseline_mcc = matthews_corrcoef(y_test, y_pred_baseline)
    
    # Tuned metrics
    tuned_precision = precision_score(y_test, y_pred_tuned, average='weighted', zero_division=0)
    tuned_recall = recall_score(y_test, y_pred_tuned, average='weighted', zero_division=0)
    tuned_f1 = f1_score(y_test, y_pred_tuned, average='weighted', zero_division=0)
    tuned_mcc = matthews_corrcoef(y_test, y_pred_tuned)
    
    # Create comparison DataFrame
    comparison_data = {
        'Model': [f'{baseline_model_name} (Baseline)', f'{baseline_model_name} (Tuned)'],
        'Accuracy': [baseline_accuracy, tuned_accuracy],
        'Precision': [baseline_precision, tuned_precision],
        'Recall': [baseline_recall, tuned_recall],
        'F1 Score': [baseline_f1, tuned_f1],
        'MCC': [baseline_mcc, tuned_mcc]
    }
    
    comparison_df = pd.DataFrame(comparison_data)
    
    print(f"\n📊 Detailed Performance Comparison:")
    print(comparison_df.to_string(index=False))
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('📈 Final Model Evaluation: Baseline vs Tuned', fontsize=16, fontweight='bold')
    
    # Accuracy comparison
    axes[0, 0].bar(['Baseline', 'Tuned'], [baseline_accuracy, tuned_accuracy], 
                  color=['lightcoral', 'lightgreen'], alpha=0.8)
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].set_title('🎯 Accuracy Comparison', fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Add value labels
    for i, acc in enumerate([baseline_accuracy, tuned_accuracy]):
        axes[0, 0].text(i, acc + 0.005, f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')
    
    # All metrics comparison
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'MCC']
    baseline_metrics = [baseline_accuracy, baseline_precision, baseline_recall, baseline_f1, baseline_mcc]
    tuned_metrics = [tuned_accuracy, tuned_precision, tuned_recall, tuned_f1, tuned_mcc]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    axes[0, 1].bar(x - width/2, baseline_metrics, width, label='Baseline', color='lightcoral', alpha=0.8)
    axes[0, 1].bar(x + width/2, tuned_metrics, width, label='Tuned', color='lightgreen', alpha=0.8)
    axes[0, 1].set_xlabel('Metrics')
    axes[0, 1].set_ylabel('Score')
    axes[0, 1].set_title('📊 All Metrics Comparison', fontweight='bold')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(metrics)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Confusion matrices
    cm_baseline = confusion_matrix(y_test, y_pred_baseline)
    cm_tuned = confusion_matrix(y_test, y_pred_tuned)
    
    # Baseline confusion matrix
    im1 = axes[0, 2].imshow(cm_baseline, interpolation='nearest', cmap=plt.cm.Blues)
    axes[0, 2].set_title('Baseline Confusion Matrix', fontweight='bold')
    axes[0, 2].set_xlabel('Predicted')
    axes[0, 2].set_ylabel('Actual')
    
    # Add text annotations
    thresh = cm_baseline.max() / 2.
    for i in range(cm_baseline.shape[0]):
        for j in range(cm_baseline.shape[1]):
            axes[0, 2].text(j, i, format(cm_baseline[i, j], 'd'),
                           ha="center", va="center",
                           color="white" if cm_baseline[i, j] > thresh else "black")
    
    # Tuned confusion matrix
    im2 = axes[1, 0].imshow(cm_tuned, interpolation='nearest', cmap=plt.cm.Greens)
    axes[1, 0].set_title('Tuned Confusion Matrix', fontweight='bold')
    axes[1, 0].set_xlabel('Predicted')
    axes[1, 0].set_ylabel('Actual')
    
    # Add text annotations
    thresh = cm_tuned.max() / 2.
    for i in range(cm_tuned.shape[0]):
        for j in range(cm_tuned.shape[1]):
            axes[1, 0].text(j, i, format(cm_tuned[i, j], 'd'),
                           ha="center", va="center",
                           color="white" if cm_tuned[i, j] > thresh else "black")
    
    # Classification reports
    print(f"\n📋 Baseline Classification Report:")
    print(classification_report(y_test, y_pred_baseline, target_names=['Class 1', 'Class 2', 'Class 3']))
    
    print(f"\n📋 Tuned Classification Report:")
    print(classification_report(y_test, y_pred_tuned, target_names=['Class 1', 'Class 2', 'Class 3']))
    
    # Performance improvement visualization
    improvements = [
        tuned_accuracy - baseline_accuracy,
        tuned_precision - baseline_precision,
        tuned_recall - baseline_recall,
        tuned_f1 - baseline_f1,
        tuned_mcc - baseline_mcc
    ]
    
    colors = ['green' if imp >= 0 else 'red' for imp in improvements]
    axes[1, 1].bar(metrics, improvements, color=colors, alpha=0.7)
    axes[1, 1].set_ylabel('Improvement')
    axes[1, 1].set_title('📈 Performance Improvements', fontweight='bold')
    axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[1, 1].grid(True, alpha=0.3)
    
    # Add improvement labels
    for i, imp in enumerate(improvements):
        axes[1, 1].text(i, imp + (0.002 if imp >= 0 else -0.002), 
                       f'{imp:+.4f}', ha='center', va='bottom' if imp >= 0 else 'top', 
                       fontweight='bold')
    
    # Model complexity comparison (simplified)
    baseline_complexity = len(baseline_model.get_params())
    tuned_complexity = len(tuned_model.get_params())
    
    axes[1, 2].bar(['Baseline', 'Tuned'], [baseline_complexity, tuned_complexity], 
                    color=['lightcoral', 'lightgreen'], alpha=0.8)
    axes[1, 2].set_ylabel('Number of Parameters')
    axes[1, 2].set_title('⚖️ Model Complexity', fontweight='bold')
    axes[1, 2].grid(True, alpha=0.3)
    
    for i, comp in enumerate([baseline_complexity, tuned_complexity]):
        axes[1, 2].text(i, comp + 1, f'{comp}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Final recommendation
    print(f"\n🏆 FINAL MODEL RECOMMENDATION:")
    print("=" * 50)
    
    if tuned_accuracy > baseline_accuracy:
        print(f"✅ RECOMMENDED: Tuned Model")
        print(f"📊 Better accuracy: {tuned_accuracy:.4f} vs {baseline_accuracy:.4f}")
        print(f"📈 Improvement: {(tuned_accuracy - baseline_accuracy):+.4f}")
        print(f"🔧 Optimized parameters: {tuned_params}")
        final_model = tuned_model
        final_model_name = f"{baseline_model_name} (Tuned)"
        final_accuracy = tuned_accuracy
    else:
        print(f"⚠️ RECOMMENDED: Baseline Model")
        print(f"📊 Tuning did not improve performance")
        print(f"📊 Baseline accuracy: {baseline_accuracy:.4f}")
        print(f"📊 Tuned accuracy: {tuned_accuracy:.4f}")
        print(f"📈 Difference: {(tuned_accuracy - baseline_accuracy):+.4f}")
        final_model = baseline_model
        final_model_name = baseline_model_name
        final_accuracy = baseline_accuracy
    
    print(f"\n💡 Key Insights:")
    print(f"• Hyperparameter tuning {'improved' if tuned_accuracy > baseline_accuracy else 'did not improve'} performance")
    print(f"• Best model: {final_model_name}")
    print(f"• Final accuracy: {final_accuracy:.4f}")
    print(f"• Training approach: SMOTE-balanced + {'tuned' if tuned_accuracy > baseline_accuracy else 'baseline'}")
    
else:
    print("❌ No models available for final comparison!")
    final_model = None
    final_model_name = None
    final_accuracy = None

print(f"\n✅ Model evaluation complete!")

## 🏆 **Champion Model Identification - Advanced Selection Framework**

### 🎯 **Objective**
Identify the true champion model using comprehensive, multi-dimensional evaluation criteria beyond just accuracy.

### 🔥 **Why Advanced Champion Selection Matters**

| **Simple Selection** | **Advanced Selection** |
|---------------------|------------------------|
| ✅ Single metric (accuracy) | ✅ **Multi-criteria evaluation** |
| ⚠️ Ignores statistical significance | ✅ **Statistical validation** |
| ⚠️ No production considerations | ✅ **Production readiness assessment** |
| ⚠️ Risk of overfitting | ✅ **Robustness testing** |
| ❌ Limited insights | ✅ **Comprehensive decision framework** |

### 📋 **Champion Selection Framework**
1. **Statistical Significance**: Paired t-tests and confidence intervals
2. **Robustness Testing**: Cross-validation stability analysis
3. **Production Metrics**: Inference speed, memory usage, complexity
4. **Business Impact**: Error cost analysis, interpretability
5. **Risk Assessment**: Overfitting detection, variance analysis

In [0]:
# Advanced Champion Model Identification
print("🏆 Advanced Champion Model Identification...")

if len(results_df) > 0:
    print("🔍 Using comprehensive framework to identify true champion model...")
    
    # 1. Statistical Significance Testing
    print("\n📊 1. Statistical Significance Analysis")
    
    from scipy import stats
    import time
    
    # Get top 5 models for detailed analysis
    top_5_models = results_sorted.head(5)
    
    statistical_results = []
    
    for idx, (_, model_row) in enumerate(top_5_models.iterrows()):
        model_name = model_row['model_name']
        model = trained_models[model_name]
        
        # Perform multiple cross-validation runs for statistical significance
        cv_scores = []
        cv_times = []
        
        for cv_run in range(10):  # 10 CV runs for statistical significance
            start_time = time.time()
            
            # Manual 5-fold cross-validation
            from sklearn.model_selection import StratifiedKFold
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + cv_run)
            fold_scores = []
            
            for train_idx, val_idx in skf.split(X_train_balanced_scaled, y_train_balanced):
                X_train_fold, X_val_fold = X_train_balanced_scaled[train_idx], X_train_balanced_scaled[val_idx]
                y_train_fold, y_val_fold = y_train_balanced.iloc[train_idx], y_train_balanced.iloc[val_idx]
                
                # Handle ensemble models differently
                if 'Voting' in model_name or 'Stacking' in model_name:
                    # For ensemble models, create a new instance with the same structure
                    if 'Voting' in model_name:
                        model_copy = VotingClassifier(
                            estimators=model.estimators,
                            voting=model.voting
                        )
                    else:  # Stacking
                        model_copy = StackingClassifier(
                            estimators=model.estimators,
                            final_estimator=model.final_estimator,
                            cv=model.cv
                        )
                elif 'Gaussian Process' in model_name:
                    # Handle Gaussian Process with kernel parameter
                    try:
                        kernel_params = model.get_params()
                        model_copy = GaussianProcessClassifier(
                            random_state=42,
                            kernel=kernel_params.get('kernel', 1.0 * RBF(1.0))
                        )
                    except Exception as e:
                        print(f"⚠️ Using fallback for Gaussian Process: {str(e)}")
                        model_copy = GaussianProcessClassifier(random_state=42)
                else:
                    # For non-ensemble models, use get_params() with fallback
                    try:
                        model_params = model.get_params()
                        # Filter out problematic parameters for certain models
                        filtered_params = {}
                        for key, value in model_params.items():
                            if not key.startswith(('rf__', 'gb__', 'lr__', 'svm__')):  # Skip ensemble-specific params
                                filtered_params[key] = value
                        model_copy = type(model)(**filtered_params)
                    except Exception as e:
                        # Fallback: use default constructor with key parameters
                        print(f"⚠️ Using fallback for {model_name}: {str(e)}")
                        model_copy = type(model)(random_state=42)
                
                model_copy.fit(X_train_fold, y_train_fold)
                fold_pred = model_copy.predict(X_val_fold)
                fold_score = accuracy_score(y_val_fold, fold_pred)
                fold_scores.append(fold_score)
            
            cv_scores.append(np.mean(fold_scores))
            cv_times.append(time.time() - start_time)
        
        mean_cv = np.mean(cv_scores)
        std_cv = np.std(cv_scores)
        mean_time = np.mean(cv_times)
        
        # 95% confidence interval
        ci_lower = mean_cv - 1.96 * (std_cv / np.sqrt(10))
        ci_upper = mean_cv + 1.96 * (std_cv / np.sqrt(10))
        
        statistical_results.append({
            'model_name': model_name,
            'mean_cv_score': mean_cv,
            'std_cv_score': std_cv,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'mean_training_time': mean_time,
            'test_accuracy': model_row['accuracy'],
            'test_f1': model_row['f1_score']
        })
    
    stat_df = pd.DataFrame(statistical_results)
    
    print("📈 Statistical Analysis Results:")
    print(stat_df[['model_name', 'mean_cv_score', 'std_cv_score', 'ci_lower', 'ci_upper', 'mean_training_time']].round(4))
    
    # 2. Overlapping Confidence Intervals Analysis
    print("\n🔍 2. Confidence Interval Overlap Analysis")
    
    best_model_idx = stat_df['mean_cv_score'].idxmax()
    best_model_stat = stat_df.iloc[best_model_idx]
    
    overlapping_models = []
    non_overlapping_models = []
    
    for idx, row in stat_df.iterrows():
        if idx != best_model_idx:
            # Check if confidence intervals overlap
            if (row['ci_lower'] <= best_model_stat['ci_upper'] and 
                row['ci_upper'] >= best_model_stat['ci_lower']):
                overlapping_models.append(row['model_name'])
            else:
                non_overlapping_models.append(row['model_name'])
    
    print(f"🏆 Best model (statistical): {best_model_stat['model_name']}")
    print(f"📊 CV Score: {best_model_stat['mean_cv_score']:.4f} ± {best_model_stat['std_cv_score']:.4f}")
    print(f"📊 95% CI: [{best_model_stat['ci_lower']:.4f}, {best_model_stat['ci_upper']:.4f}]")
    
    if non_overlapping_models:
        print(f"✅ Statistically better than: {', '.join(non_overlapping_models)}")
    else:
        print(f"⚠️ No statistically significant difference from other top models")
    
    # 3. Production Metrics Analysis
    print("\n⚡ 3. Production Metrics Analysis")
    
    production_metrics = []
    
    for idx, (_, model_row) in enumerate(top_5_models.iterrows()):
        model_name = model_row['model_name']
        model = trained_models[model_name]
        
        # Inference speed test
        inference_times = []
        for _ in range(100):  # 100 inference runs
            start_time = time.time()
            _ = model.predict(X_test_scaled[:1])  # Single prediction
            inference_times.append(time.time() - start_time)
        
        mean_inference_time = np.mean(inference_times) * 1000  # Convert to ms
        
        # Memory usage estimation (simplified)
        import pickle
        model_size = len(pickle.dumps(model)) / (1024 * 1024)  # MB
        
        # Model complexity (number of parameters)
        try:
            n_params = len(model.get_params())
        except:
            n_params = 50  # Default for complex models
        
        production_metrics.append({
            'model_name': model_name,
            'inference_time_ms': mean_inference_time,
            'model_size_mb': model_size,
            'n_parameters': n_params,
            'cv_score': stat_df.loc[stat_df['model_name'] == model_name, 'mean_cv_score'].iloc[0]
        })
    
    prod_df = pd.DataFrame(production_metrics)
    
    print("⚡ Production Metrics:")
    print(prod_df[['model_name', 'inference_time_ms', 'model_size_mb', 'n_parameters']].round(4))
    
    # 4. Business Impact Analysis
    print("\n💼 4. Business Impact Analysis")
    
    # Simulate business impact based on confusion matrices
    business_impact = []
    
    for idx, (_, model_row) in enumerate(top_5_models.iterrows()):
        model_name = model_row['model_name']
        model = trained_models[model_name]
        
        y_pred = model.predict(X_test_scaled)
        cm = confusion_matrix(y_test, y_pred)
        
        # Simulate business costs (example: wine classification)
        # Class 1: Premium wine (high value)
        # Class 2: Standard wine (medium value) 
        # Class 3: Budget wine (low value)
        
        # Cost of misclassification (penalty matrix)
        cost_matrix = np.array([
            [0, 2, 5],    # Actual Class 1 misclassified as 2, 3
            [1, 0, 3],    # Actual Class 2 misclassified as 1, 3
            [2, 1, 0]     # Actual Class 3 misclassified as 1, 2
        ])
        
        # Calculate total business cost
        total_cost = 0
        total_samples = len(y_test)
        
        for i in range(3):  # actual classes
            for j in range(3):  # predicted classes
                if i != j:  # misclassifications only
                    total_cost += cm[i, j] * cost_matrix[i, j]
        
        cost_per_sample = total_cost / total_samples
        
        # Calculate business value (inverse of cost)
        business_value = 1 / (1 + cost_per_sample)
        
        business_impact.append({
            'model_name': model_name,
            'misclassification_cost': total_cost,
            'cost_per_sample': cost_per_sample,
            'business_value': business_value,
            'accuracy': model_row['accuracy']
        })
    
    business_df = pd.DataFrame(business_impact)
    
    print("💼 Business Impact Analysis:")
    print(business_df[['model_name', 'cost_per_sample', 'business_value', 'accuracy']].round(4))
    
    # 5. Risk Assessment (Overfitting Detection)
    print("\n⚠️ 5. Risk Assessment - Overfitting Detection")
    
    risk_assessment = []
    
    for idx, (_, model_row) in enumerate(top_5_models.iterrows()):
        model_name = model_row['model_name']
        model = trained_models[model_name]
        
        # Get corresponding statistical data
        stat_data = stat_df[stat_df['model_name'] == model_name].iloc[0]
        
        # Overfitting indicators
        cv_score = stat_data['mean_cv_score']
        test_score = model_row['accuracy']
        overfitting_gap = cv_score - test_score
        
        # Variance indicator (std of CV scores)
        variance_indicator = stat_data['std_cv_score']
        
        # Model complexity risk
        try:
            n_params = len(model.get_params())
        except:
            n_params = 50  # Default for complex models
        complexity_risk = n_params / 100  # Normalized
        
        # Overall risk score (0-1, lower is better)
        risk_score = (abs(overfitting_gap) + variance_indicator + complexity_risk) / 3
        
        risk_assessment.append({
            'model_name': model_name,
            'cv_score': cv_score,
            'test_score': test_score,
            'overfitting_gap': overfitting_gap,
            'variance_indicator': variance_indicator,
            'complexity_risk': complexity_risk,
            'overall_risk_score': risk_score
        })
    
    risk_df = pd.DataFrame(risk_assessment)
    
    print("⚠️ Risk Assessment:")
    print(risk_df[['model_name', 'overfitting_gap', 'variance_indicator', 'overall_risk_score']].round(4))
    
    # 6. Comprehensive Champion Selection
    print("\n🏆 6. Comprehensive Champion Selection")
    
    # Combine all metrics into final scoring
    champion_candidates = []
    
    for idx, (_, model_row) in enumerate(top_5_models.iterrows()):
        model_name = model_row['model_name']
        
        # Get data from all analyses
        stat_data = stat_df[stat_df['model_name'] == model_name].iloc[0]
        prod_data = prod_df[prod_df['model_name'] == model_name].iloc[0]
        business_data = business_df[business_df['model_name'] == model_name].iloc[0]
        risk_data = risk_df[risk_df['model_name'] == model_name].iloc[0]
        
        # Calculate composite scores (0-1 scale, higher is better)
        
        # Performance score (40% weight)
        performance_score = stat_data['mean_cv_score']
        
        # Stability score (20% weight) - lower variance is better
        stability_score = 1 - (stat_data['std_cv_score'] / stat_data['mean_cv_score'])
        
        # Production score (20% weight) - lower time and size is better
        production_score = (1 - (prod_data['inference_time_ms'] / prod_data['inference_time_ms'].max())) * 0.5 + \
                          (1 - (prod_data['model_size_mb'] / prod_data['model_size_mb'].max())) * 0.5
        
        # Business value score (15% weight)
        business_score = business_data['business_value']
        
        # Risk score (5% weight) - lower risk is better
        risk_score = 1 - risk_data['overall_risk_score']
        
        # Final composite score
        final_score = (performance_score * 0.4 + 
                      stability_score * 0.2 + 
                      production_score * 0.2 + 
                      business_score * 0.15 + 
                      risk_score * 0.05)
        
        champion_candidates.append({
            'model_name': model_name,
            'performance_score': performance_score,
            'stability_score': stability_score,
            'production_score': production_score,
            'business_score': business_score,
            'risk_score': risk_score,
            'final_score': final_score,
            'accuracy': model_row['accuracy']
        })
    
    champion_df = pd.DataFrame(champion_candidates)
    champion_df_sorted = champion_df.sort_values('final_score', ascending=False)
    
    print("🏆 FINAL CHAMPION MODEL RANKINGS:")
    print("=" * 80)
    
    for i, (_, row) in enumerate(champion_df_sorted.iterrows(), 1):
        print(f"{i}. {row['model_name']:<25} | Final Score: {row['final_score']:.4f} | Accuracy: {row['accuracy']:.4f}")
        print(f"   Performance: {row['performance_score']:.3f} | Stability: {row['stability_score']:.3f} | Production: {row['production_score']:.3f}")
        print(f"   Business: {row['business_score']:.3f} | Risk: {row['risk_score']:.3f}")
        print()
    
    # Identify champion
    champion_model = champion_df_sorted.iloc[0]
    champion_name = champion_model['model_name']
    champion_score = champion_model['final_score']
    
    print(f"🏆 CHAMPION MODEL: {champion_name}")
    print(f"📊 Final Composite Score: {champion_score:.4f}")
    print(f"🎯 Accuracy: {champion_model['accuracy']:.4f}")
    print(f"⚖️ Performance Score: {champion_model['performance_score']:.4f}")
    print(f"🛡️ Stability Score: {champion_model['stability_score']:.4f}")
    print(f"⚡ Production Score: {champion_model['production_score']:.4f}")
    print(f"💼 Business Score: {champion_model['business_score']:.4f}")
    print(f"⚠️ Risk Score: {champion_model['risk_score']:.4f}")
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('🏆 Comprehensive Champion Model Analysis', fontsize=16, fontweight='bold')
    
    # Final scores comparison
    axes[0, 0].barh(champion_df_sorted['model_name'].str[:15], champion_df_sorted['final_score'], 
                    color='gold', alpha=0.8)
    axes[0, 0].set_xlabel('Final Composite Score')
    axes[0, 0].set_title('🏆 Champion Model Rankings', fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Component scores radar chart
    categories = ['Performance', 'Stability', 'Production', 'Business', 'Risk']
    champion_components = [champion_model['performance_score'], champion_model['stability_score'], 
                          champion_model['production_score'], champion_model['business_score'], 
                          champion_model['risk_score']]
    
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    champion_components += champion_components[:1]  # Complete the circle
    angles += angles[:1]
    
    axes[0, 1].plot(angles, champion_components, 'o-', linewidth=2, color='red', label='Champion')
    axes[0, 1].fill(angles, champion_components, alpha=0.25, color='red')
    axes[0, 1].set_xticks(angles[:-1])
    axes[0, 1].set_xticklabels(categories)
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].set_title('🎯 Champion Model Profile', fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Statistical significance visualization
    axes[0, 2].errorbar(range(len(stat_df)), stat_df['mean_cv_score'], 
                      yerr=stat_df['std_cv_score'], fmt='o', capsize=5, capthick=2)
    axes[0, 2].set_xticks(range(len(stat_df)))
    axes[0, 2].set_xticklabels([name[:10] for name in stat_df['model_name']], rotation=45)
    axes[0, 2].set_ylabel('CV Score')
    axes[0, 2].set_title('📊 Statistical Significance', fontweight='bold')
    axes[0, 2].grid(True, alpha=0.3)
    
    # Production metrics
    axes[1, 0].scatter(prod_df['inference_time_ms'], prod_df['model_size_mb'], 
                      s=100, alpha=0.7, c=range(len(prod_df)), cmap='viridis')
    for i, row in prod_df.iterrows():
        axes[1, 0].annotate(row['model_name'][:8], 
                           (row['inference_time_ms'], row['model_size_mb']), 
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
    axes[1, 0].set_xlabel('Inference Time (ms)')
    axes[1, 0].set_ylabel('Model Size (MB)')
    axes[1, 0].set_title('⚡ Production Performance', fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Business impact
    axes[1, 1].bar(business_df['model_name'].str[:15], business_df['business_value'], 
                  color='green', alpha=0.8)
    axes[1, 1].set_ylabel('Business Value')
    axes[1, 1].set_title('💼 Business Impact', fontweight='bold')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(True, alpha=0.3)
    
    # Risk assessment
    axes[1, 2].barh(risk_df['model_name'].str[:15], risk_df['overall_risk_score'], 
                   color='orange', alpha=0.8)
    axes[1, 2].set_xlabel('Risk Score (lower is better)')
    axes[1, 2].set_title('⚠️ Risk Assessment', fontweight='bold')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Set the champion model for next steps
    final_champion_model = trained_models[champion_name]
    
    print(f"\n✅ Champion model identification complete!")
    print(f"🏆 Champion: {champion_name}")
    print(f"📊 This model will be used for Unity Catalog registration and serving.")
    
else:
    print("❌ No models available for champion identification!")
    final_champion_model = None
    champion_name = None

## 🚀 Model Registration in Unity Catalog

### 🎯 **Objective**
Register the best sklearn model in Unity Catalog for production deployment.

### 🏪 **Why Unity Catalog?**
- **Centralized Governance**: Single source of truth for all models
- **Access Control**: Fine-grained permissions on model access
- **Versioning**: Track model evolution and rollbacks
- **Lineage**: Understand model provenance and dependencies
- **Compliance**: Meet regulatory requirements for model management

### 📋 **Registration Steps**
1. **Get Model URI** from MLflow run
2. **Register Model** in Unity Catalog
3. **Set Permissions** for appropriate access
4. **Add Documentation** for model consumers

In [0]:
# Model Registration in Unity Catalog
print("🏪 Registering Champion Model in Unity Catalog...")

# Check if we have a champion model from the advanced selection
if champion_name is not None and final_champion_model is not None:
    print(f"🏆 Using champion model: {champion_name}")
    
    # Use the champion model information
    selected_model_name = champion_name
    selected_model = final_champion_model
    selected_accuracy = champion_df[champion_df['model_name'] == champion_name]['accuracy'].iloc[0]
    
    # Find the corresponding run information from training results
    champion_run_info = next((run for run in training_results if run['model_name'] == selected_model_name), None)
    
    if champion_run_info is not None:
        best_run_id = champion_run_info['run_id']
        print(f"🔗 Found MLflow run: {best_run_id}")
    else:
        print("⚠️ No MLflow run found for champion model, creating new registration...")
        # Log the champion model to MLflow first
        with mlflow.start_run(run_name=f"{champion_name}_Champion_Registration") as run:
            signature = infer_signature(X_train_balanced_scaled, final_champion_model.predict(X_test_scaled))
            input_example = X_train_balanced_scaled[:5]
            
            mlflow.sklearn.log_model(
                final_champion_model,
                name="model",
                signature=signature,
                input_example=input_example
            )
            
            # Log champion model metrics
            y_pred = final_champion_model.predict(X_test_scaled)
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
            recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
            f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
            
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("champion_score", champion_df[champion_df['model_name'] == champion_name]['final_score'].iloc[0])
            
            best_run_id = run.info.run_id
            print(f"📦 Created new MLflow run: {best_run_id}")
    
    # Construct model URI
    model_uri = f"runs:/{best_run_id}/model"
    print(f"📦 Model URI: {model_uri}")
    
    # Try Unity Catalog registration first, then fallback to legacy
    print(f"\n🔧 Attempting Unity Catalog registration...")
    
    # Unity Catalog model name (three-level namespace)
    uc_model_name = "dsa.model.wine"
    
    try:
        # Register the champion model in Unity Catalog
        model_version = mlflow.register_model(
            model_uri=model_uri,
            name=uc_model_name,
            tags={
                "model_type": "sklearn",
                "algorithm": champion_name.lower().replace(" ", "_"),
                "framework": "hybrid_spark_sklearn",
                "purpose": "wine_classification",
                "environment": "production",
                "selection_method": "champion_model",
                "champion_score": str(round(champion_df[champion_df['model_name'] == champion_name]['final_score'].iloc[0], 4))
            }
        )
        
        print(f"✅ Champion model registered successfully in Unity Catalog!")
        print(f"🏪 Model Name: {uc_model_name}")
        print(f"🔢 Version: {model_version.version}")
        print(f"🏆 Algorithm: {champion_name}")
        print(f"📊 Accuracy: {selected_accuracy:.4f}")
        print(f"🏅 Champion Score: {champion_df[champion_df['model_name'] == champion_name]['final_score'].iloc[0]:.4f}")
        print(f"🔗 MLflow Run: {best_run_id}")
        
        registered_model_name = uc_model_name
        registration_method = "unity_catalog"
        
    except Exception as e:
        print(f"⚠️ Unity Catalog registration failed: {str(e)}")
        print(f"� Falling back to legacy Workspace Model Registry...")
        
        # Fallback to legacy workspace model registry
        legacy_model_name = "wine_production_hybrid"
        
        try:
            model_version = mlflow.register_model(
                model_uri=model_uri,
                name=legacy_model_name,
                tags={
                    "model_type": "sklearn",
                    "algorithm": champion_name.lower().replace(" ", "_"),
                    "framework": "hybrid_spark_sklearn",
                    "purpose": "wine_classification",
                    "environment": "production",
                    "selection_method": "champion_model",
                    "champion_score": str(round(champion_df[champion_df['model_name'] == champion_name]['final_score'].iloc[0], 4))
                }
            )
            
            print(f"✅ Champion model registered successfully in Legacy Registry!")
            print(f"🏪 Model Name: {legacy_model_name}")
            print(f"🔢 Version: {model_version.version}")
            print(f"🏆 Algorithm: {champion_name}")
            print(f"📊 Accuracy: {selected_accuracy:.4f}")
            print(f"🏅 Champion Score: {champion_df[champion_df['model_name'] == champion_name]['final_score'].iloc[0]:.4f}")
            print(f"🔗 MLflow Run: {best_run_id}")
            
            registered_model_name = legacy_model_name
            registration_method = "legacy_workspace"
            
        except Exception as e2:
            print(f"❌ Both registration methods failed!")
            print(f"❌ Unity Catalog Error: {str(e)}")
            print(f"❌ Legacy Registry Error: {str(e2)}")
            print(f"\n� Troubleshooting:")
            print(f"   1. Check Unity Catalog permissions and setup")
            print(f"   2. Verify catalog 'dsa' and schema 'catalog' exist")
            print(f"   3. Ensure MLflow tracking URI is configured correctly")
            print(f"   4. Contact workspace admin for registry permissions")
            model_version = None
            registration_method = None

elif best_model_name is not None and best_model is not None:
    print(f"📊 Using best model from basic comparison: {best_model_name}")
    
    # Get the best model's run information
    best_run_info = next((run for run in training_results if run['model_name'] == best_model_name), None)
    
    if best_run_info is not None:
        best_run_id = best_run_info['run_id']
        print(f"🔗 Found MLflow run: {best_run_id}")
        
        # Construct model URI
        model_uri = f"runs:/{best_run_id}/model"
        print(f"📦 Model URI: {model_uri}")
        
        # Try Unity Catalog registration first, then fallback to legacy
        print(f"\n🔧 Attempting Unity Catalog registration...")
        
        # Unity Catalog model name (three-level namespace)
        uc_model_name = "dsa.catalog.wine.production_hybrid"
        
        try:
            # Register the model in Unity Catalog
            model_version = mlflow.register_model(
                model_uri=model_uri,
                name=uc_model_name,
                tags={
                    "model_type": "sklearn",
                    "algorithm": best_model_name.lower().replace(" ", "_"),
                    "framework": "hybrid_spark_sklearn",
                    "purpose": "wine_classification",
                    "environment": "production",
                    "selection_method": "best_accuracy"
                }
            )
            
            print(f"✅ Model registered successfully in Unity Catalog!")
            print(f"🏪 Model Name: {uc_model_name}")
            print(f"🔢 Version: {model_version.version}")
            print(f"� MLflow Run: {best_run_id}")
            
            registered_model_name = uc_model_name
            registration_method = "unity_catalog"
            
        except Exception as e:
            print(f"⚠️ Unity Catalog registration failed: {str(e)}")
            print(f"🔄 Falling back to legacy Workspace Model Registry...")
            
            # Fallback to legacy workspace model registry
            legacy_model_name = "wine_production_hybrid"
            
            try:
                model_version = mlflow.register_model(
                    model_uri=model_uri,
                    name=legacy_model_name,
                    tags={
                        "model_type": "sklearn",
                        "algorithm": best_model_name.lower().replace(" ", "_"),
                        "framework": "hybrid_spark_sklearn",
                        "purpose": "wine_classification",
                        "environment": "production",
                        "selection_method": "best_accuracy"
                    }
                )
                
                print(f"✅ Model registered successfully in Legacy Registry!")
                print(f"🏪 Model Name: {legacy_model_name}")
                print(f"🔢 Version: {model_version.version}")
                print(f"🔗 MLflow Run: {best_run_id}")
                
                registered_model_name = legacy_model_name
                registration_method = "legacy_workspace"
                
            except Exception as e2:
                print(f"❌ Both registration methods failed!")
                print(f"❌ Unity Catalog Error: {str(e)}")
                print(f"❌ Legacy Registry Error: {str(e2)}")
                model_version = None
                registration_method = None
    else:
        print("❌ No MLflow run found for best model")

else:
    print("❌ No model available for registration!")
    model_version = None
    registered_model_name = None
    registration_method = None

# Get model details and update description
if model_version is not None and registered_model_name is not None:
    try:
        client = mlflow.MlflowClient()
        model_details = client.get_model_version(registered_model_name, model_version.version)
        
        print(f"\n📋 Model Details:")
        print(f"📅 Created: {model_details.creation_timestamp}")
        print(f"👤 Created By: {model_details.user_id}")
        print(f"🏷️  Status: {model_details.current_stage}")
        print(f"📝 Description: {model_details.description}")
        print(f"🔧 Registry Type: {registration_method}")
        
        # Add comprehensive model description
        if registration_method == "unity_catalog":
            description = (f"Champion wine classification model registered in Unity Catalog. "
                         f"{(champion_name if champion_name else best_model_name)} with {(selected_accuracy if 'selected_accuracy' in locals() else best_accuracy):.4f} accuracy. "
                         f"Selected via {'champion model framework' if champion_name else 'best accuracy comparison'}. "
                         f"Trained on SMOTE-balanced data with {len(X_train_balanced)} samples.")
        else:
            description = (f"Champion wine classification model registered in Legacy Workspace Registry. "
                         f"{(champion_name if champion_name else best_model_name)} with {(selected_accuracy if 'selected_accuracy' in locals() else best_accuracy):.4f} accuracy. "
                         f"Selected via {'champion model framework' if champion_name else 'best accuracy comparison'}. "
                         f"Trained on SMOTE-balanced data with {len(X_train_balanced)} samples.")
        
        client.update_model_version(
            name=registered_model_name,
            version=model_version.version,
            description=description
        )
        
        print(f"\n✅ Model description updated!")
        
        # Store registration info for next steps
        registered_model_info = {
            'name': registered_model_name,
            'version': model_version.version,
            'algorithm': champion_name if champion_name else best_model_name,
            'accuracy': selected_accuracy if 'selected_accuracy' in locals() else best_accuracy,
            'run_id': best_run_id if 'best_run_id' in locals() else None,
            'registration_method': registration_method
        }
        
        print(f"\n🎉 Registration Summary:")
        print(f"🏪 Registry: {registration_method}")
        print(f"📝 Model Name: {registered_model_name}")
        print(f"🔢 Version: {model_version.version}")
        print(f"🏆 Algorithm: {registered_model_info['algorithm']}")
        print(f"📊 Accuracy: {registered_model_info['accuracy']:.4f}")
        
        if registration_method == "legacy_workspace":
            print(f"\n💡 Note: Using Legacy Workspace Model Registry")
            print(f"💡 For production workloads, consider configuring Unity Catalog")
            print(f"💡 Contact your workspace admin for Unity Catalog setup")
        
    except Exception as e:
        print(f"⚠️ Could not retrieve model details: {str(e)}")
        registered_model_info = None
else:
    registered_model_info = None

print(f"\n✅ Model registration process complete!")

## 🌐 **8. Model Serving Endpoint Deployment**

### 🎯 **Objective**
Deploy the registered model as a real-time serving endpoint.

### 🚀 **Why sklearn for Serving?**

The key advantage of our hybrid approach becomes evident here:

| **Aspect** | **sklearn Model** | **Spark MLlib Model** |
|------------|-------------------|------------------------|
| **Cold Start Time** | ✅ **~1-2 seconds** | ❌ **~30-60 seconds** |
| **Memory Footprint** | ✅ **~10-50 MB** | ❌ **~500MB-1GB** |
| **Initialization** | ✅ **Simple** | ❌ **Spark context needed** |
| **Scalability** | ✅ **Easy** | ⚠️ Complex |
| **Monitoring** | ✅ **Built-in** | ⚠️ Limited |

### 📋 **Deployment Steps**
1. **Create Serving Endpoint** with appropriate configuration
2. **Configure Scaling** for expected traffic
3. **Set Up Monitoring** and alerts
4. **Test Endpoint** functionality

In [0]:
# Deploy Model Serving Endpoint with Advanced Configuration
print("🌐 Deploying Model Serving Endpoint with Advanced Configuration...")

# Check if we have a registered model
if registered_model_info is not None:
    endpoint_name = "wine-production-hybrid-endpoint"
    registered_model_name = registered_model_info['name']
    model_version_num = registered_model_info['version']
    
    try:
        # Import advanced serving components
        from datetime import timedelta, datetime
        from databricks.sdk.service.serving import (
            EndpointCoreConfigInput, 
            ServedEntityInput, 
            AiGatewayConfig,
            AiGatewayRateLimit,
            AiGatewayRateLimitKey,
            AiGatewayRateLimitRenewalPeriod,
            AiGatewayInferenceTableConfig,
            AiGatewayUsageTrackingConfig,
            EmailNotifications,
            EndpointTag,
            ServingModelWorkloadType
        )
        
        # Check if endpoint already exists
        existing_endpoints = workspace_client.serving_endpoints.list()
        existing = None
        for ep in existing_endpoints:
            if ep.name == endpoint_name:
                existing = ep
                break
        
        # Check if endpoint is currently being updated
        if existing is not None:
            print(f"🔍 Checking endpoint status...")
            endpoint_details = workspace_client.serving_endpoints.get(endpoint_name)
            
            if endpoint_details.state.config_update:
                print(f"⏳ Endpoint is currently being updated. Waiting for completion...")
                
                # Wait for the current update to complete
                max_wait_time = 300  # 5 minutes max wait
                wait_interval = 10  # Check every 10 seconds
                waited_time = 0
                
                while endpoint_details.state.config_update and waited_time < max_wait_time:
                    print(f"⏳ Still updating... ({waited_time}s elapsed)")
                    time.sleep(wait_interval)
                    waited_time += wait_interval
                    
                    # Refresh endpoint status
                    endpoint_details = workspace_client.serving_endpoints.get(endpoint_name)
                
                if endpoint_details.state.config_update:
                    print(f"⚠️ Update still in progress after {max_wait_time}s. Proceeding with caution...")
                else:
                    print(f"✅ Previous update completed successfully!")
        
        if existing is None:
            # Create new endpoint with advanced features
            print(f"🔧 Creating new endpoint with advanced features: {endpoint_name}")
            
            endpoint = workspace_client.serving_endpoints.create_and_wait(
                name=endpoint_name,
                config=EndpointCoreConfigInput(
                    served_entities=[
                        ServedEntityInput(
                            name="wine-hybrid-champion",
                            entity_name=registered_model_name,
                            entity_version=str(model_version_num),
                            workload_size="Small",
                            workload_type=ServingModelWorkloadType.CPU,
                            scale_to_zero_enabled=True,
                            environment_vars={
                                "PYSPARK_PYTHON": "/databricks/python3/bin/python3",
                                "PYSPARK_DRIVER_PYTHON": "/databricks/python3/bin/python3"
                            }
                        )
                    ]
                ),
                ai_gateway=AiGatewayConfig(
                    rate_limits=[
                        AiGatewayRateLimit(
                            renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
                            calls=200,  # 200 calls per minute
                            key=AiGatewayRateLimitKey.ENDPOINT
                        )
                    ],
                    usage_tracking_config=AiGatewayUsageTrackingConfig(enabled=True)
                ),
                description=f"Hybrid Spark + sklearn wine classification endpoint. {(champion_name if champion_name else best_model_name)} with {registered_model_info['accuracy']:.4f} accuracy. Selected via {'champion model framework' if champion_name else 'best accuracy comparison'}.",
                tags=[
                    EndpointTag(key="project", value="wine_classification"),
                    EndpointTag(key="framework", value="hybrid_spark_sklearn"),
                    EndpointTag(key="environment", value="production"),
                    EndpointTag(key="selection_method", value="champion_model" if champion_name else "best_accuracy"),
                    EndpointTag(key="algorithm", value=(champion_name if champion_name else best_model_name).lower().replace(" ", "_")),
                    EndpointTag(key="team", value="ml-platform"),
                    EndpointTag(key="cost-center", value="ds-0000")
                ],
                email_notifications=EmailNotifications(
                    on_update_failure=["admin@company.com"]  # Replace with actual email
                ),
                route_optimized=False,
                timeout=timedelta(minutes=30)
            )
            
            print(f"✅ Advanced endpoint created successfully!")
            print(f"🌐 Endpoint Name: {endpoint.name}")
            print(f"🔗 Endpoint URL: {endpoint.endpoint_url}")
            print(f"📊 Status: {endpoint.state.ready}")
            print(f"🏆 Model: {(champion_name if champion_name else best_model_name)}")
            print(f"📊 Accuracy: {registered_model_info['accuracy']:.4f}")
            print(f"🔐 Rate Limit: 200 calls/minute")
            print(f"⏱️ Timeout: 30 minutes")
            print(f"📧 Notifications: Enabled for failures")
            
        else:
            # Update existing endpoint - only supported parameters
            print(f"🔄 Updating existing endpoint with basic features: {endpoint_name}")
            
            # For updating, only use supported parameters
            updated_endpoint = workspace_client.serving_endpoints.update_config_and_wait(
                name=endpoint_name,
                served_entities=[
                    ServedEntityInput(
                        name="wine-hybrid-champion",
                        entity_name=registered_model_name,
                        entity_version=str(model_version_num),
                        workload_size="Small",
                        workload_type=ServingModelWorkloadType.CPU,
                        scale_to_zero_enabled=True,
                        environment_vars={
                            "PYSPARK_PYTHON": "/databricks/python3/bin/python3",
                            "PYSPARK_DRIVER_PYTHON": "/databricks/python3/bin/python3"
                        }
                    )
                ]
            )
            
            print(f"✅ Endpoint updated successfully!")
            print(f"🔗 Endpoint URL: {updated_endpoint.endpoint_url}")
            print(f"📊 Status: {updated_endpoint.state.ready}")
            print(f"🏆 Model: {(champion_name if champion_name else best_model_name)}")
            print(f"📊 Accuracy: {registered_model_info['accuracy']:.4f}")
            print(f"⚠️ Note: Advanced features (AI Gateway, tags, notifications) only available for new endpoints")
        
        # Store endpoint information
        endpoint_info = {
            'name': endpoint_name,
            'url': endpoint.endpoint_url if existing is None else updated_endpoint.endpoint_url,
            'model_name': registered_model_name,
            'model_version': model_version_num,
            'algorithm': champion_name if champion_name else best_model_name,
            'accuracy': registered_model_info['accuracy'],
            'timeout': '30 minutes',
            'notifications': True if existing is None else False
        }
        
        # Enhanced endpoint testing
        print(f"\n🧪 Enhanced Endpoint Testing...")
        
        try:
            import requests
            import json
            import time
            
            # Prepare multiple test samples
            test_samples = X_test_scaled[:3].tolist()  # First 3 test samples
            
            print(f"🧪 Testing with {len(test_samples)} samples...")
            
            for i, sample_data in enumerate(test_samples):
                # Create request payload
                payload = {
                    "inputs": [
                        {
                            "name": "features",
                            "shape": [1, len(feature_columns)],
                            "datatype": "FLOAT64",
                            "data": [sample_data]
                        }
                    ]
                }
                
                # Make prediction request with timing
                start_time = time.time()
                response = requests.post(
                    url=endpoint_info['url'],
                    headers={"Content-Type": "application/json"},
                    data=json.dumps(payload)
                )
                response_time = time.time() - start_time
                
                if response.status_code == 200:
                    prediction_result = response.json()
                    predicted_class = prediction_result['predictions'][0]
                    actual_class = y_test.iloc[i]
                    
                    print(f"✅ Test {i+1}: Predicted {predicted_class}, Actual {actual_class}, Correct: {predicted_class == actual_class}, Time: {response_time:.3f}s")
                    
                else:
                    print(f"⚠️ Test {i+1} failed with status {response.status_code}")
                    print(f"Response: {response.text[:200]}...")
            
            # Test rate limiting (quick burst test)
            print(f"\n🔐 Testing rate limiting...")
            burst_start = time.time()
            successful_calls = 0
            
            for i in range(5):  # Test 5 rapid calls
                try:
                    response = requests.post(
                        url=endpoint_info['url'],
                        headers={"Content-Type": "application/json"},
                        data=json.dumps(payload)
                    )
                    if response.status_code == 200:
                        successful_calls += 1
                except:
                    pass
            
            burst_time = time.time() - burst_start
            print(f"📊 Burst test: {successful_calls}/5 calls successful in {burst_time:.3f}s")
            
        except Exception as e:
            print(f"⚠️ Could not perform enhanced testing: {str(e)}")
            print("💡 This might be due to endpoint not being fully ready yet")
        
        # Display configuration summary
        print(f"\n🚀 Endpoint Configuration Summary:")
        if existing is None:
            print(f"🔐 Rate Limiting: 200 calls per minute")
            print(f"📊 Usage Tracking: Enabled")
            print(f"📧 Email Notifications: Configured for failures")
        else:
            print(f"📊 Basic Configuration: Updated model version")
            print(f"⚠️ Advanced Features: Only available for new endpoints")
            print(f"💡 To get advanced features, delete and recreate the endpoint")
        
        print(f"⏱️ Timeout: 30 minutes")
        print(f"🔧 Environment: Production with PySpark vars")
        print(f"💰 Scale-to-Zero: Enabled for cost optimization")
        
    except Exception as e:
        print(f"❌ Error deploying endpoint: {str(e)}")
        print("💡 Common issues:")
        print("   - Endpoint currently being updated by another process")
        print("   - Insufficient permissions for serving endpoints")
        print("   - Model not properly registered in Unity Catalog")
        print("   - Workspace serving endpoints not enabled")
        print(f"   - Model {registered_model_name} v{model_version_num} not found")
        print("   - Endpoint configuration syntax error")
        
        print("\n🔧 Troubleshooting:")
        print("   1. Wait a few minutes and try again (update in progress)")
        print("   2. Check Unity Catalog permissions")
        print("   3. Verify model registration")
        print("   4. Contact workspace admin")
        print("   5. Check model version exists")
        print("   6. Try basic endpoint deployment")
        
        # Try basic endpoint deployment without any advanced features
        print(f"\n🔄 Trying basic endpoint deployment...")
        
        try:
            basic_endpoint = workspace_client.serving_endpoints.update_config_and_wait(
                name=endpoint_name,
                served_entities=[
                    ServedEntityInput(
                        name="wine-hybrid-champion",
                        entity_name=registered_model_name,
                        entity_version=str(model_version_num),
                        workload_size="Small",
                        workload_type=ServingModelWorkloadType.CPU,
                        scale_to_zero_enabled=True
                    )
                ]
            )
            
            print(f"✅ Basic endpoint deployed successfully!")
            print(f"🔗 Endpoint URL: {basic_endpoint.endpoint_url}")
            print(f"📊 Status: {basic_endpoint.state.ready}")
            
            endpoint_info = {
                'name': endpoint_name,
                'url': basic_endpoint.endpoint_url,
                'model_name': registered_model_name,
                'model_version': model_version_num,
                'algorithm': champion_name if champion_name else best_model_name,
                'accuracy': registered_model_info['accuracy']
            }
            
        except Exception as e2:
            print(f"❌ Basic endpoint deployment also failed: {str(e2)}")
            endpoint_info = None

else:
    print("❌ No registered model available for endpoint deployment!")
    endpoint_info = None

print(f"\n💡 Features Available:")
if endpoint_info is not None:
    if 'existing' not in locals() or existing is None:
        print(f"🔐 Rate Limiting: 200 calls/minute")
        print(f"📊 Usage Tracking: Enabled")
        print(f"📧 Notifications: Alert on deployment failures")
    else:
        print(f"📊 Basic Configuration: Model updated")
        print(f"⚠️ Advanced Features: Create new endpoint for full features")
    
print(f"⏱️ Custom Timeout: Prevents hanging requests")
print(f"🏷️ Rich Metadata: Better organization and governance")
print(f"💰 Cost Optimization: Scale-to-zero when idle")

print(f"\n📋 Production Checklist:")
print(f"1. 🧪 Test endpoint with various input formats")
print(f"2. 📊 Set up monitoring dashboards")
print(f"3. 🔔 Configure alert thresholds")
print(f"4. 📚 Create API documentation")
print(f"5. 💰 Monitor costs and usage patterns")
print(f"6. 🔐 Review security and access controls")

if endpoint_info is not None:
    print(f"\n🌐 Endpoint Information:")
    print(f"📝 Name: {endpoint_info['name']}")
    print(f"🔗 URL: {endpoint_info['url']}")
    print(f"🏆 Model: {endpoint_info['algorithm']}")
    print(f"📊 Accuracy: {endpoint_info['accuracy']:.4f}")
    print(f"🔢 Version: {endpoint_info['model_version']}")

print(f"\n✅ Model serving deployment complete!")

## 🧪 **Endpoint Testing and Inference**

### 🎯 **Objective**
Test the deployed endpoint with sample data to ensure it's working correctly.

### 🔄 **Testing Strategy**
1. **Health Check**: Verify endpoint is accessible
2. **Sample Inference**: Test with known data
3. **Performance Test**: Check response times
4. **Error Handling**: Test edge cases

In [0]:
endpoint_info['url']

In [0]:
# Test the Deployed Endpoint
print("🧪 Testing the Deployed Endpoint...")

# First, get the current endpoint information
endpoint_name = "wine-production-hybrid-endpoint"

try:
    # Get current endpoint details
    current_endpoints = workspace_client.serving_endpoints.list()
    endpoint_info = None
    
    for ep in current_endpoints:
        if ep.name == endpoint_name:
            # Extract host without protocol for clean URL construction
            host = workspace_client.config.host
            if host.startswith('https://'):
                host = host[8:]  # Remove 'https://'
            elif host.startswith('http://'):
                host = host[7:]   # Remove 'http://'
            
            endpoint_info = {
                'name': ep.name,
                'url': f"https://{host}/serving-endpoints/{ep.name}/invocations",  # Correct Databricks format
                'algorithm': 'Hybrid Model',  # Will be updated if we have registered info
                'accuracy': 0.0  # Will be updated if we have registered info
            }
            break
    
    if endpoint_info is not None:
        print(f"✅ Found endpoint: {endpoint_info['name']}")
        print(f"🔗 URL: {endpoint_info['url']}")
        
        # Try to get model information if available
        try:
            if 'registered_model_info' in locals() and registered_model_info is not None:
                endpoint_info['algorithm'] = registered_model_info['algorithm']
                endpoint_info['accuracy'] = registered_model_info['accuracy']
                print(f"🏆 Model: {endpoint_info['algorithm']}")
                print(f"📊 Accuracy: {endpoint_info['accuracy']:.4f}")
        except:
            print(f"⚠️ Model information not available")
            
    else:
        print(f"❌ Endpoint not found: {endpoint_name}")
        print("💡 Please run the endpoint deployment cell first")
        
except Exception as e:
    print(f"❌ Error getting endpoint information: {str(e)}")
    endpoint_info = None

# Check if endpoint is available
if endpoint_info is not None:
    # Import required libraries
    import os
    import requests
    import json
    import time
    import pandas as pd
    
    def create_tf_serving_json(data):
        """
        Create TensorFlow Serving JSON format from data.
        Handles both dictionary and numpy array inputs.
        """
        return {'inputs': {name: data[name].tolist() for name in data.keys()} if isinstance(data, dict) else data.tolist()}
    
    def score_model(dataset, endpoint_url):
        """
        Score model using Databricks serving endpoint.
        Handles both DataFrame and other data formats.
        """
        # Get Databricks token
        databricks_token = dbutils.secrets.get(scope = "admin", key = "pat")
        if not databricks_token:
            # Try to get token from workspace client
            try:
                databricks_token = workspace_client.config.token
            except:
                databricks_token = os.environ.get("DATABRICKS_TOKEN")

        
        # Prepare headers
        headers = {
            'Authorization': f'Bearer {databricks_token}', 
            'Content-Type': 'application/json'
        }
        
        # Convert dataset to appropriate format
        if isinstance(dataset, pd.DataFrame):
            ds_dict = {'dataframe_split': dataset.to_dict(orient='split')}
        else:
            ds_dict = create_tf_serving_json(dataset)
        
        # Make request
        data_json = json.dumps(ds_dict, allow_nan=True)
        response = requests.request(method='POST', headers=headers, url=endpoint_url, data=data_json)
        
        if response.status_code != 200:
            raise Exception(f'Request failed with status {response.status_code}, {response.text}')
        
        return response.json()
    
    # Sample test data (same format as training data with engineered features)
    test_samples = [
        {
            "ALCOHOL": 13.72,
            "MALICACID": 1.43,
            "ASH": 2.5,
            "ALCALINITY_OF_ASH": 16.7,
            "MAGNESIUM": 108,
            "TOTAL_PHENOLS": 3.4,
            "FLAVANOIDS": 3.67,
            "NONFLAVANOID_PHENOLS": 0.19,
            "PROANTHOCYANINS": 2.04,
            "COLOR_INTENSITY": 6.8,
            "HUE": 0.89,
            "DILUTED_WINES": 2.87,
            "PROLINE": 1285,
            "ALCOHOL_ACIDITY_RATIO": 13.72 / 1.43,  # Engineered feature
            "TOTAL_PHENOL_CONTENT": 3.4 + 3.67,      # Engineered feature
            "COLOR_INDEX": 6.8 * 0.89                # Engineered feature
        },
        {
            "ALCOHOL": 12.37,
            "MALICACID": 0.94,
            "ASH": 1.36,
            "ALCALINITY_OF_ASH": 10.6,
            "MAGNESIUM": 88,
            "TOTAL_PHENOLS": 1.98,
            "FLAVANOIDS": 0.57,
            "NONFLAVANOID_PHENOLS": 0.28,
            "PROANTHOCYANINS": 0.42,
            "COLOR_INTENSITY": 1.95,
            "HUE": 1.05,
            "DILUTED_WINES": 1.82,
            "PROLINE": 520,
            "ALCOHOL_ACIDITY_RATIO": 12.37 / 0.94,
            "TOTAL_PHENOL_CONTENT": 1.98 + 0.57,
            "COLOR_INDEX": 1.95 * 1.05
        },
        {
            "ALCOHOL": 11.27,
            "MALICACID": 0.84,
            "ASH": 2.18,
            "ALCALINITY_OF_ASH": 17.8,
            "MAGNESIUM": 67,
            "TOTAL_PHENOLS": 2.34,
            "FLAVANOIDS": 0.49,
            "NONFLAVANOID_PHENOLS": 0.39,
            "PROANTHOCYANINS": 0.71,
            "COLOR_INTENSITY": 4.41,
            "HUE": 1.12,
            "DILUTED_WINES": 1.90,
            "PROLINE": 765,
            "ALCOHOL_ACIDITY_RATIO": 11.27 / 0.84,
            "TOTAL_PHENOL_CONTENT": 2.34 + 0.49,
            "COLOR_INDEX": 4.41 * 1.12
        }
    ]
    
    try:
        # Test endpoint using improved scoring function
        endpoint_url = endpoint_info['url']
        
        print(f"🌐 Testing endpoint: {endpoint_url}")
        print(f"🏆 Model: {endpoint_info['algorithm']}")
        print(f"📊 Expected Accuracy: {endpoint_info['accuracy']:.4f}")
        print(f"🔐 Authentication: Bearer token (DATABRICKS_TOKEN)")
        
        # Test each sample with timing
        results = []
        
        for i, sample in enumerate(test_samples, 1):
            print(f"\n🧪 Testing Sample {i}:")
            
            # Create DataFrame for sample (ensures proper column ordering)
            sample_df = pd.DataFrame([sample])
            sample_df = sample_df[feature_columns]  # Ensure correct column order
            
            # Make prediction with timing
            start_time = time.time()
            result = score_model(sample_df, endpoint_url)
            response_time = time.time() - start_time
            
            prediction = result['predictions'][0]
            
            print(f"✅ Sample {i} - Predicted Class: {prediction}")
            print(f"⚡ Response Time: {response_time:.3f} seconds")
            print(f"📊 Feature Preview: {list(sample.values())[:5]}...")
            
            results.append({
                'sample': i,
                'predicted_class': prediction,
                'response_time': response_time,
                'status': 'success'
            })
        
        # Summary statistics
        successful_tests = [r for r in results if r['status'] == 'success']
        
        if successful_tests:
            avg_response_time = sum(r['response_time'] for r in successful_tests) / len(successful_tests)
            predictions = [r['predicted_class'] for r in successful_tests]
            
            print(f"\n📊 Testing Summary:")
            print(f"✅ Successful Tests: {len(successful_tests)}/{len(test_samples)}")
            print(f"⚡ Average Response Time: {avg_response_time:.3f} seconds")
            print(f"🎯 Predictions: {predictions}")
            
            # Check response time performance
            if avg_response_time < 0.1:
                print(f"🚀 Excellent performance: <100ms average response time")
            elif avg_response_time < 0.5:
                print(f"✅ Good performance: <500ms average response time")
            else:
                print(f"⚠️ Performance concern: {avg_response_time:.3f}s average response time")
        
        print(f"\n✅ Endpoint testing complete!")
        
    except Exception as e:
        print(f"❌ Error testing endpoint: {str(e)}")
        
        # Check for specific error types
        if "401" in str(e):
            print(f"🔐 Authentication error - Check DATABRICKS_TOKEN environment variable")
        elif "403" in str(e):
            print(f"🚫 Permission error - Check endpoint access permissions")
        elif "404" in str(e):
            print(f"🔍 Not found - Check endpoint URL")
        elif "429" in str(e):
            print(f"⏱️ Rate limited - Too many requests")
        
        print("\n💡 Testing with local model for validation...")
        
        # Fallback: Test with local champion model
        print("\n🔄 Testing with local champion model...")
        
        for i, sample in enumerate(test_samples, 1):
            # Convert to DataFrame and scale
            sample_df = pd.DataFrame([sample])
            
            # Ensure columns are in the same order as training
            sample_df = sample_df[feature_columns]
            sample_scaled = sklearn_scaler.transform(sample_df)
            
            # Make prediction with champion model
            prediction = final_champion_model.predict(sample_scaled)[0]
            probabilities = final_champion_model.predict_proba(sample_scaled)[0]
            
            print(f"\n🧪 Sample {i} (Local Champion Model):")
            print(f"✅ Predicted Class: {prediction}")
            print(f"📊 Probabilities: Class 1: {probabilities[0]:.3f}, Class 2: {probabilities[1]:.3f}, Class 3: {probabilities[2]:.3f}")
            print(f"🏆 Model: {champion_name if champion_name else best_model_name}")

else:
    print("❌ No endpoint available for testing!")
    print("💡 Please complete model registration and deployment steps first")

## 📊 **Production Monitoring and Maintenance**

### 🎯 **Objective**
Set up monitoring and maintenance strategies for the production model.

### 📈 **Monitoring Metrics**
- **Model Performance**: Accuracy, precision, recall over time
- **Endpoint Health**: Response times, error rates
- **Data Drift**: Feature distribution changes
- **Usage Patterns**: Request volume, user behavior

### 🔧 **Maintenance Tasks**
- **Model Retraining**: Periodic updates with new data
- **Performance Validation**: A/B testing new versions
- **Dependency Updates**: Keep libraries current
- **Security Audits**: Regular access reviews

In [0]:
# Production Monitoring Setup
print("📊 Setting up Production Monitoring...")

# Create monitoring dashboard mockup
monitoring_data = {
    "timestamp": pd.date_range(start="2024-01-01", periods=30, freq="D"),
    "accuracy": np.random.normal(best_accuracy, 0.02, 30),
    "response_time_ms": np.random.normal(50, 10, 30),
    "requests_per_hour": np.random.normal(100, 20, 30),
    "error_rate": np.random.uniform(0, 0.02, 30)
}

monitoring_df = pd.DataFrame(monitoring_data)

# Create monitoring visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Model Accuracy Over Time
axes[0, 0].plot(monitoring_df['timestamp'], monitoring_df['accuracy'], 'b-', linewidth=2)
axes[0, 0].axhline(y=best_accuracy, color='r', linestyle='--', label=f'Target: {best_accuracy:.3f}')
axes[0, 0].set_title('🎯 Model Accuracy Over Time', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Response Time
axes[0, 1].plot(monitoring_df['timestamp'], monitoring_df['response_time_ms'], 'g-', linewidth=2)
axes[0, 1].axhline(y=100, color='r', linestyle='--', label='SLA: 100ms')
axes[0, 1].set_title('⚡ Response Time', fontweight='bold')
axes[0, 1].set_ylabel('Response Time (ms)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Request Volume
axes[1, 0].plot(monitoring_df['timestamp'], monitoring_df['requests_per_hour'], 'purple', linewidth=2)
axes[1, 0].set_title('📈 Request Volume', fontweight='bold')
axes[1, 0].set_ylabel('Requests per Hour')
axes[1, 0].grid(True, alpha=0.3)

# Error Rate
axes[1, 1].plot(monitoring_df['timestamp'], monitoring_df['error_rate'], 'r-', linewidth=2)
axes[1, 1].axhline(y=0.05, color='orange', linestyle='--', label='Alert: 5%')
axes[1, 1].set_title('❌ Error Rate', fontweight='bold')
axes[1, 1].set_ylabel('Error Rate')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Monitoring recommendations
print("\n📊 Production Monitoring Recommendations:")
print("\n🔍 Key Metrics to Monitor:")
print("1. 📈 Model Performance:")
print("   - Accuracy drift detection")
print("   - Prediction confidence distribution")
print("   - Class balance monitoring")

print("\n2. ⚡ Endpoint Performance:")
print("   - Response time SLA (<100ms)")
print("   - Error rate (<1%)")
print("   - Availability (>99.9%)")

print("\n3. 📊 Usage Patterns:")
print("   - Request volume trends")
print("   - Peak usage hours")
print("   - Geographic distribution")

print("\n🔧 Maintenance Schedule:")
print("• Daily: Performance metrics review")
print("• Weekly: Data drift analysis")
print("• Monthly: Model retraining evaluation")
print("• Quarterly: Security and dependency updates")

print("\n🚨 Alert Thresholds:")
print("• Accuracy drop > 5%")
print("• Response time > 200ms")
print("• Error rate > 2%")
print("• Request volume spike > 3x normal")